# ForgeryGuard AI — Phase 2: Detection, Segmentation, Integration and Deployment

**AI251P Applied Computer Vision Capstone — M4, M5 and M8**

This notebook is the executable Phase 2 record for ForgeryGuard AI. It preserves the corrected M1 source-group split, applies the documented M2/M3 exclusions, fine-tunes and evaluates the detector and segmenter, integrates both models into an end-to-end pipeline, verifies an ONNX export, exposes a tested FastAPI contract, benchmarks the complete system, and produces the evidence required for the Phase 2 report.

هذا الدفتر هو السجل التنفيذي للمرحلة الثانية. يحافظ على تقسيم المرحلة الأولى المصحح، ويطبق الاستبعادات الموثقة في المرحلتين الثانية والثالثة، ثم يدرب ويقيّم نماذج الكشف والتقسيم، ويدمجهما في مسار واحد مع واجهة برمجية واختبار تصدير وقياس أداء وأدلة جاهزة للتقرير.

### Required Kaggle inputs

- The project dataset containing the corrected `dataset_split_portable.csv`, `tampering_bounding_boxes.csv`, and MSCOCO original images.
- The DEFACTO Copy-Move dataset containing forged images and probe masks.
- Internet access for official pretrained weights, unless the weights are attached as Kaggle inputs.
- A GPU accelerator for training and benchmarking.

### Execution rule

Run the cells in order. The notebook deliberately pauses at the manual annotation review until the reviewer records a real decision. Validation data selects thresholds; test data is evaluated after the operating points are fixed. Source images are never deleted or redistributed.

تشغّل الخلايا بالترتيب. يتوقف التنفيذ عند المراجعة اليدوية حتى يسجل المراجع نتيجة فعلية. تستخدم بيانات التحقق لاختيار الحدود، ثم تستخدم بيانات الاختبار للتقييم النهائي بعد تثبيت الإعدادات. لا تُحذف الصور الأصلية ولا يعاد توزيعها.


## Before running — Kaggle settings and W&B credential

1. Open **Kaggle → Notebook settings** and select a GPU accelerator.
2. Enable Internet, or attach `yolo11n.pt` and `resnet18-f37072fd.pth` as inputs.
3. Open **Add-ons → Secrets** and create a secret named `WANDB_API_KEY` using the personal or team key from Weights & Biases. Grant this notebook access to the secret.
4. Do not paste the key into a code cell, notebook output, GitHub, or the report.

قبل التشغيل، فعّل معالج الرسوميات والإنترنت. أضف مفتاح تتبع التجارب في أسرار كاجل باسم `WANDB_API_KEY`. لا يُكتب المفتاح داخل الكود ولا يظهر في المخرجات أو المستودع.

FastAPI does not require an external API key in this notebook. The HTTP contract is tested locally. A public tunnel is optional for the presentation and is not required for the reproducible notebook run.

لا تحتاج واجهة الخدمة إلى مفتاح خارجي داخل هذا الدفتر. يتم اختبار عقد الواجهة محليًا، ويمكن لاحقًا استخدام رابط عام للعرض إذا لزم.


## Shared — Step 1: Environment and required packages

Install only missing runtime packages, then report the execution environment. Package versions are captured again in the final reproducibility manifest.

نثبت المكتبات الناقصة فقط، ثم نسجل بيئة التشغيل. تُحفظ الإصدارات الفعلية لاحقًا ضمن سجل إعادة التشغيل.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "ultralytics": "ultralytics>=8.3,<9",
    "wandb": "wandb>=0.18,<1",
    "yaml": "PyYAML>=6,<7",
    "onnx": "onnx>=1.17,<2",
    "onnxruntime": "onnxruntime>=1.20,<2",
    "fastapi": "fastapi>=0.115,<1",
    "multipart": "python-multipart>=0.0.12,<1",
    "uvicorn": "uvicorn>=0.34,<1",
    "httpx": "httpx>=0.27,<1",
    "psutil": "psutil>=6,<8",
}

missing = [spec for module, spec in required_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("All required packages are already available.")


## Shared — Step 2: Fixed configuration and run identity

The configuration fixes seeds, model sizes, training schedules, validation grids, benchmark conditions, and acceptance criteria before test evaluation. The notebook supports checkpoint reuse through explicit paths, but the official run uses one recorded configuration.

تُثبت هذه الخلية البذور وأحجام النماذج وإعدادات التدريب وشبكات اختيار الحدود وشروط القياس ومعايير القبول قبل تقييم الاختبار. يمكن استكمال العمل من نقاط حفظ صريحة دون تغيير التقسيم أو الإعدادات.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from io import BytesIO
import base64
import hashlib
import importlib.metadata as metadata
import json
import math
import os
import platform
import random
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from tqdm.auto import tqdm
import yaml

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
from IPython.display import display, Markdown

TEAM_NAME = "ForgeryGuard_AI"
RUN_OWNER = "Abdullah Alhuthali and Mishari Alghamdi"

# Optional checkpoint reuse. Leave empty for the official full training run.
M4_CHECKPOINT_OVERRIDE = ""
M5_CHECKPOINT_OVERRIDE = ""

# These four source groups were excluded by the documented M2/M3 decoded-pixel audit.
M2_M3_CONFLICT_GROUPS = {
    "000000417134", "000000537055", "000000448046", "000000469870"
}

EXPECTED_M1_MANIFEST_SHA256 = "42d9c01f7588366ed5e2a327da4c2cca5253ea0a4989167827ec8df386f0a9c6"
LEGACY_M1_MANIFEST_SHA256 = "11f55a0a459ec0fb814f691fd9732bb7f3c6a648030ec9d6dfda3694bf5a3bec"
EXPECTED_M1_SEMANTIC_SHA256 = "b86d01060a6b7cdf0cf1b64c66fb10f1b60839a2b9d6b8f1ab89f0b51b851dc5"
M1_NEAR_DUPLICATE_EXCLUDED_GROUP = "000000396188"

CFG = {
    "seed": 251,
    "project": "ForgeryGuard_AI_Phase2",
    "team": TEAM_NAME,
    "run_owner": RUN_OWNER,
    "class_name": "copy_move_region",
    "mask_threshold": 127,
    "m4_model": "yolo11n.pt",
    "m4_imgsz": 640,
    "m4_batch": 16,
    "m4_epochs": 30,
    "m4_patience": 8,
    "m4_workers": 2,
    "m4_conf_grid": [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50],
    "m4_nms_grid": [0.40, 0.50, 0.60],
    "m5_imgsz": 256,
    "m5_batch": 32,
    "m5_epochs": 20,
    "m5_patience": 5,
    "m5_workers": 0,
    "m5_lr": 3e-4,
    "m5_weight_decay": 1e-4,
    "m5_crop_padding": 0.15,
    "m5_negative_ratio": 0.25,
    "m5_threshold_grid": [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80],
    "mask_ap_iou_thresholds": [round(x, 2) for x in np.arange(0.50, 0.96, 0.05)],
    "audit_samples": {"train": 36, "val": 12, "test": 12},
    "audit_page_size": 12,
    "m8_warmup_runs": 5,
    "m8_benchmark_images": 50,
    "onnx_opset": 17,
    "acceptance": {
        "m4_min_test_map50": 0.70,
        "m4_min_test_recall": 0.85,
        "m5_min_test_dice": 0.75,
        "m5_min_test_iou": 0.65,
        "onnx_max_abs_difference": 0.001,
        "m8_max_p95_ms": 1000.0,
        "m8_min_throughput_images_per_second": 2.0,
    },
}

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/ForgeryGuard_AI_Phase2_M4_M5_M8")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    return value

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(to_builtin(value), indent=2, ensure_ascii=False, allow_nan=False),
        encoding="utf-8",
    )
    temporary.replace(path)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

seed_all(CFG["seed"])
write_json(OUTPUT_ROOT / "config.json", CFG)

print("Run owner:", RUN_OWNER)
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Enable a Kaggle GPU before training and benchmarking.")
print("Output:", OUTPUT_ROOT)


## Shared — Step 3: W&B credential and tracking mode

The credential is read from the Kaggle secret `WANDB_API_KEY`; its value is never printed. Online tracking is required for the final submission link. If the secret is missing, the notebook continues in offline mode and marks W&B as pending in the final checklist.

يُقرأ المفتاح من أسرار كاجل دون طباعته. التتبع المباشر مطلوب للحصول على رابط التسليم. إذا لم يوجد المفتاح يستمر التنفيذ محليًا ويظهر بند التتبع كمتطلب غير مكتمل في الفحص النهائي.


In [ ]:
import wandb

WANDB_API_KEY_AVAILABLE = False
try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    if wandb_key:
        wandb.login(key=wandb_key, relogin=True)
        WANDB_API_KEY_AVAILABLE = True
        del wandb_key
except Exception as exc:
    print("W&B secret is not available in this session:", type(exc).__name__)

WANDB_MODE = "online" if WANDB_API_KEY_AVAILABLE else "offline"
print("W&B mode:", WANDB_MODE)
if WANDB_MODE == "offline":
    print("ACTION REQUIRED: Add WANDB_API_KEY in Kaggle Secrets before the final run, or sync the saved offline run.")


## Shared — Step 4: Predeclared acceptance criteria

The proposal targets are frozen before model selection and final test evaluation. Failure to meet a target is reported as a limitation and influences the final go/no-go decision; it is not hidden by changing the target after observing test results.

تُثبت أهداف المقترح قبل اختيار النماذج وتقييم الاختبار. عدم تحقيق هدف يُسجل كقيد ويؤثر في قرار الاستخدام النهائي، ولا يتم تعديل الهدف بعد رؤية النتائج.


In [ ]:
ACCEPTANCE_RECORD = {
    "frozen_before_test_evaluation": True,
    "source": "ForgeryGuard AI approved project proposal",
    "criteria": CFG["acceptance"],
    "selection_policy": {
        "m4": "Select confidence and NMS IoU on validation data by target recall, then F1.",
        "m5": "Select the binary mask threshold on validation Dice and IoU.",
        "test": "Evaluate final operating points without test-set tuning.",
    },
}
write_json(OUTPUT_ROOT / "acceptance_criteria.json", ACCEPTANCE_RECORD)
display(pd.DataFrame([CFG["acceptance"]]).T.rename(columns={0: "target"}))


## Shared — Step 5: Resolve and verify attached data

Resolve the corrected M1 manifest by its committed SHA-256 value. If only the verified legacy manifest is attached, reproduce the documented near-duplicate exclusion and verify the resulting canonical content. Then locate boxes, original images, forged images, and masks.

نحدد ملف التقسيم المصحح ببصمته المعتمدة. إذا كانت النسخة القديمة الموثقة هي المتاحة، نعيد تطبيق استبعاد التشابه القريب ونتحقق من المحتوى الناتج، ثم نحدد المربعات والصور والأقنعة.


In [ ]:
def candidates_named(root, name):
    return sorted(p for p in root.rglob(name) if p.is_file())

def choose_file_by_sha(name, expected_sha=None):
    matches = candidates_named(INPUT_ROOT, name)
    if not matches:
        raise FileNotFoundError(f"Required input not found: {name}")
    if expected_sha:
        exact = [p for p in matches if sha256_file(p) == expected_sha]
        if len(exact) == 1:
            return exact[0]
        if len(exact) > 1:
            print(f"Multiple identical verified copies of {name}; using {exact[0]}")
            return exact[0]
        observed = {str(p): sha256_file(p) for p in matches}
        raise FileNotFoundError(
            f"The verified {name} was not found. Expected SHA-256 {expected_sha}. "
            f"Observed: {observed}"
        )
    hashes = {sha256_file(p) for p in matches}
    if len(hashes) == 1:
        return matches[0]
    raise RuntimeError(f"Multiple different copies of {name}: {matches}")

def choose_m1_manifest():
    matches = candidates_named(INPUT_ROOT, "dataset_split_portable.csv")
    if not matches:
        raise FileNotFoundError("Required input not found: dataset_split_portable.csv")
    observed = {p: sha256_file(p) for p in matches}
    corrected = [p for p, digest in observed.items() if digest == EXPECTED_M1_MANIFEST_SHA256]
    legacy = [p for p, digest in observed.items() if digest == LEGACY_M1_MANIFEST_SHA256]
    if corrected:
        return corrected[0], "corrected_manifest_attached"
    if legacy:
        return legacy[0], "legacy_manifest_verified_correction_will_be_rebuilt"
    raise FileNotFoundError(
        "No verified M1 manifest was found. Expected the corrected or documented legacy SHA-256. "
        f"Observed: { {str(path): digest for path, digest in observed.items()} }"
    )

def choose_image_dir(pattern, suffixes, min_files=100):
    choices = []
    for path in INPUT_ROOT.rglob(pattern):
        if path.is_dir():
            count = sum(1 for p in path.iterdir() if p.suffix.lower() in suffixes)
            if count >= min_files:
                choices.append((count, path))
    if not choices:
        raise FileNotFoundError(f"No image directory matched {pattern}")
    choices.sort(key=lambda item: (-item[0], str(item[1])))
    best_count, best_path = choices[0]
    tied = [p for count, p in choices if count == best_count]
    if len(tied) > 1:
        raise RuntimeError(f"Ambiguous image directories for {pattern}: {tied}")
    return best_path

MANIFEST_PATH, MANIFEST_SOURCE_STATUS = choose_m1_manifest()
BBOX_PATH = choose_file_by_sha("tampering_bounding_boxes.csv")
ORIGINAL_DIR = choose_image_dir("Original", {".jpg", ".jpeg", ".png"})
FORGED_DIR = choose_image_dir("img", {".tif", ".tiff", ".jpg", ".png"})
MASK_DIR = choose_image_dir("probe_mask", {".jpg", ".jpeg", ".png", ".tif", ".tiff"})

RESOLVED_INPUTS = {
    "manifest": str(MANIFEST_PATH),
    "manifest_sha256": sha256_file(MANIFEST_PATH),
    "manifest_source_status": MANIFEST_SOURCE_STATUS,
    "bounding_boxes": str(BBOX_PATH),
    "bounding_boxes_sha256": sha256_file(BBOX_PATH),
    "original_dir": str(ORIGINAL_DIR),
    "forged_dir": str(FORGED_DIR),
    "mask_dir": str(MASK_DIR),
}
write_json(OUTPUT_ROOT / "resolved_inputs.json", RESOLVED_INPUTS)
print(json.dumps(RESOLVED_INPUTS, indent=2))


## Shared — Step 6: Preserve the corrected split and apply documented exclusions

The corrected M1 manifest excludes the confirmed cross-split near-duplicate source group. When the verified legacy file is attached, that exact exclusion is rebuilt automatically and checked through a canonical semantic hash. Four additional source groups identified by the M2/M3 decoded-pixel conflict audit are then excluded without reassigning any remaining source group. The effective Phase 2 set contains 18,189 source pairs.

يستبعد ملف المرحلة الأولى المصحح مجموعة التشابه القريب. إذا كانت النسخة القديمة الموثقة هي المرفقة، يعاد بناء الاستبعاد نفسه تلقائيًا ويُتحقق منه ببصمة محتوى معيارية. ثم تُستبعد أربع مجموعات تعارض موثقة دون إعادة توزيع البقية، ليصبح الإجمالي 18,189 زوجًا.


In [ ]:
MANIFEST = pd.read_csv(MANIFEST_PATH, dtype=str)

required_manifest = {
    "source_group",
    "split",
    "original_filename",
    "forged_filename",
}

assert required_manifest.issubset(MANIFEST.columns)

MANIFEST["source_group"] = (
    MANIFEST["source_group"]
    .astype(str)
    .str.zfill(12)
)

assert MANIFEST[list(required_manifest)].notna().all().all()
assert MANIFEST["source_group"].is_unique
assert MANIFEST.groupby("source_group")["split"].nunique().max() == 1
assert set(MANIFEST["split"]) == {"train", "val", "test"}

# إعادة بناء تصحيح M1 عند استخدام النسخة القديمة الموثقة
SOURCE_MANIFEST_SHA256 = sha256_file(MANIFEST_PATH)

if SOURCE_MANIFEST_SHA256 == LEGACY_M1_MANIFEST_SHA256:
    assert len(MANIFEST) == 18194

    assert (
        M1_NEAR_DUPLICATE_EXCLUDED_GROUP
        in set(MANIFEST["source_group"])
    )

    MANIFEST = MANIFEST.loc[
        MANIFEST["source_group"]
        != M1_NEAR_DUPLICATE_EXCLUDED_GROUP
    ].copy().reset_index(drop=True)

    EFFECTIVE_MANIFEST_PATH = (
        OUTPUT_ROOT / "dataset_split_portable_corrected.csv"
    )

    MANIFEST.to_csv(
        EFFECTIVE_MANIFEST_PATH,
        index=False,
    )

    print(
        "Rebuilt the documented M1 near-duplicate exclusion "
        "from the verified legacy manifest."
    )

else:
    assert (
        SOURCE_MANIFEST_SHA256
        == EXPECTED_M1_MANIFEST_SHA256
    )

    EFFECTIVE_MANIFEST_PATH = MANIFEST_PATH

# التحقق من تطابق محتوى التقسيم المصحح
semantic_columns = [
    "source_group",
    "split",
    "original_filename",
    "forged_filename",
]

canonical_manifest = (
    MANIFEST[semantic_columns]
    .sort_values(semantic_columns)
    .to_csv(
        index=False,
        lineterminator="\n",
    )
)

M1_MANIFEST_SEMANTIC_SHA256 = hashlib.sha256(
    canonical_manifest.encode()
).hexdigest()

assert (
    M1_MANIFEST_SEMANTIC_SHA256
    == EXPECTED_M1_SEMANTIC_SHA256
), "The rebuilt manifest does not match the approved corrected M1 content."

assert len(MANIFEST) == 18193

# تطبيق استبعادات M2/M3 الموثقة
missing_conflicts = (
    M2_M3_CONFLICT_GROUPS
    - set(MANIFEST["source_group"])
)

assert not missing_conflicts, (
    f"Documented conflict groups are absent: {missing_conflicts}"
)

PHASE2_PAIRS = MANIFEST.loc[
    ~MANIFEST["source_group"].isin(
        M2_M3_CONFLICT_GROUPS
    )
].copy().reset_index(drop=True)

assert len(PHASE2_PAIRS) == 18189

assert (
    PHASE2_PAIRS
    .groupby("source_group")["split"]
    .nunique()
    .max()
    == 1
)

# تحميل المربعات
BBOX = pd.read_csv(
    BBOX_PATH,
    dtype={
        "source_group": str,
        "forged_filename": str,
    },
)

box_columns = {
    "forged_filename",
    "xmin",
    "ymin",
    "xmax",
    "ymax",
}

assert box_columns.issubset(BBOX.columns)

if "source_group" not in BBOX.columns:
    filename_to_group = (
        PHASE2_PAIRS
        .set_index("forged_filename")["source_group"]
    )

    BBOX["source_group"] = (
        BBOX["forged_filename"]
        .map(filename_to_group)
    )

BBOX["source_group"] = (
    BBOX["source_group"]
    .astype(str)
    .str.zfill(12)
)

for column in ["xmin", "ymin", "xmax", "ymax"]:
    BBOX[column] = pd.to_numeric(
        BBOX[column],
        errors="raise",
    )

BBOX = BBOX.loc[
    BBOX["source_group"].isin(
        set(PHASE2_PAIRS["source_group"])
    )
].copy()

# فهرسة كل مجلد مرة واحدة بدل آلاف عمليات فحص الملفات
def scan_directory_once(directory, allowed_suffixes):
    indexed = {}

    with os.scandir(directory) as entries:
        for entry in entries:
            suffix = Path(entry.name).suffix.lower()

            if entry.is_file() and suffix in allowed_suffixes:
                indexed[entry.name] = entry.path

    return indexed


print("Indexing image and mask directories once...")

original_index = scan_directory_once(
    ORIGINAL_DIR,
    {".jpg", ".jpeg", ".png"},
)

forged_index = scan_directory_once(
    FORGED_DIR,
    {".tif", ".tiff", ".jpg", ".jpeg", ".png"},
)

mask_file_index = scan_directory_once(
    MASK_DIR,
    {".jpg", ".jpeg", ".png", ".tif", ".tiff"},
)

mask_stem_index = {}

for filename, filepath in mask_file_index.items():
    stem = Path(filename).stem

    mask_stem_index.setdefault(
        stem,
        [],
    ).append(filepath)

# ربط المسارات باستخدام الفهارس
PHASE2_PAIRS["original_path"] = (
    PHASE2_PAIRS["original_filename"]
    .map(original_index)
)

PHASE2_PAIRS["forged_path"] = (
    PHASE2_PAIRS["forged_filename"]
    .map(forged_index)
)

PHASE2_PAIRS["mask_path"] = (
    PHASE2_PAIRS["forged_filename"]
    .map(
        lambda filename: mask_stem_index.get(
            Path(filename).stem,
            [None],
        )[0]
    )
)

# التحقق من المسارات دون إعادة فحص القرص
for column in [
    "original_path",
    "forged_path",
    "mask_path",
]:
    missing = PHASE2_PAIRS.loc[
        PHASE2_PAIRS[column].isna()
    ]

    assert missing.empty, (
        f"Missing paths in {column}: {len(missing)}"
    )

# التأكد من عدم وجود أكثر من قناع للصورة نفسها
retained_mask_stems = set(
    PHASE2_PAIRS["forged_filename"]
    .map(lambda filename: Path(filename).stem)
)

ambiguous_mask_stems = {
    stem: paths
    for stem, paths in mask_stem_index.items()
    if len(paths) > 1
    and stem in retained_mask_stems
}

assert not ambiguous_mask_stems, (
    "Multiple masks found for retained images: "
    f"{list(ambiguous_mask_stems)[:5]}"
)

# التحقق من وجود مربعات لكل صورة معدلة
pair_files = set(
    PHASE2_PAIRS["forged_filename"]
)

missing_box_files = (
    pair_files
    - set(BBOX["forged_filename"])
)

assert not missing_box_files, (
    "Some retained forged images have no bounding box."
)

# حفظ الجداول الفعالة
PHASE2_PAIRS.to_csv(
    OUTPUT_ROOT / "phase2_pairs.csv",
    index=False,
)

BBOX.to_csv(
    OUTPUT_ROOT / "phase2_boxes.csv",
    index=False,
)

split_counts = (
    PHASE2_PAIRS["split"]
    .value_counts()
    .reindex(["train", "val", "test"])
)

assert split_counts.to_dict() == {
    "train": 12733,
    "val": 2728,
    "test": 2728,
}

display(
    split_counts
    .rename("source_pairs")
    .to_frame()
)

print("Effective source pairs:", len(PHASE2_PAIRS))
print("M1 manifest source status:", MANIFEST_SOURCE_STATUS)
print(
    "M1 canonical semantic SHA-256:",
    M1_MANIFEST_SEMANTIC_SHA256,
)
print(
    "M2/M3 conflict exclusions:",
    sorted(M2_M3_CONFLICT_GROUPS),
)

## Shared — Step 7: Automated annotation integrity audit

Decode every retained forged image and mask, verify matching dimensions and nonempty binary masks, and confirm that every saved box is finite, ordered, and inside the image boundary. This structural audit precedes human semantic review.

نفك ترميز كل صورة معدلة وقناع، ونتحقق من تطابق الأبعاد وعدم فراغ القناع، ثم نتحقق من أن جميع المربعات رقمية ومرتبة وداخل حدود الصورة. يسبق هذا الفحص الآلي المراجعة البشرية للمعنى والمحاذاة.


In [ ]:
pair_lookup = PHASE2_PAIRS.set_index("forged_filename")
dimension_cache = {}
structural_issues = []
mask_rows = []

for row in tqdm(PHASE2_PAIRS.itertuples(index=False), total=len(PHASE2_PAIRS), desc="Annotation integrity audit"):
    with Image.open(row.forged_path) as image:
        image.load()
        width, height = image.size
    with Image.open(row.mask_path) as mask_image:
        mask_array = np.asarray(mask_image.convert("L"))
    dimension_cache[row.forged_filename] = (width, height)
    if mask_array.shape != (height, width):
        structural_issues.append((row.forged_filename, "mask_dimension_mismatch", None, None, None, None))
    binary_pixels = int((mask_array > CFG["mask_threshold"]).sum())
    if binary_pixels == 0:
        structural_issues.append((row.forged_filename, "empty_mask", None, None, None, None))
    mask_rows.append({"forged_filename": row.forged_filename, "width": width,
                      "height": height, "positive_mask_pixels": binary_pixels})

for filename, group in tqdm(BBOX.groupby("forged_filename"), desc="Bounding-box audit"):
    width, height = dimension_cache[filename]
    for row in group.itertuples(index=False):
        coordinates = np.array([row.xmin, row.ymin, row.xmax, row.ymax], dtype=float)
        if not np.isfinite(coordinates).all():
            structural_issues.append((filename, "non_finite_box", *coordinates.tolist()))
        elif not (0 <= row.xmin < row.xmax <= width and 0 <= row.ymin < row.ymax <= height):
            structural_issues.append((filename, "out_of_bounds_box", *coordinates.tolist()))

issues_frame = pd.DataFrame(structural_issues, columns=[
    "forged_filename", "issue", "xmin", "ymin", "xmax", "ymax"
])
issues_frame.to_csv(OUTPUT_ROOT / "annotation_structural_issues.csv", index=False)
pd.DataFrame(mask_rows).to_csv(OUTPUT_ROOT / "mask_integrity_audit.csv", index=False)
assert issues_frame.empty, f"Annotation integrity issues found: {len(issues_frame)}"

print("Automated annotation audit: PASS")
print("Forged images decoded:", len(dimension_cache))
print("Masks decoded and validated:", len(mask_rows))
print("Bounding-box rows validated:", len(BBOX))


## Shared — Step 8A: Mandatory visual review of boxes and masks

A deterministic stratified sample of 60 source pairs is displayed: 36 training, 12 validation, and 12 test pairs. Each row shows the original image, the forged image with the saved box, and the forged image with the probe-mask overlay. The reviewer must inspect every row for label plausibility, box coverage, mask alignment, and obvious privacy concerns.

نعرض عينة ثابتة من 60 زوجًا: 36 من التدريب و12 من التحقق و12 من الاختبار. يعرض كل صف الصورة الأصلية والصورة المعدلة مع المربع والصورة المعدلة مع القناع. يجب فحص جميع الصفوف من حيث صحة العلاقة وتغطية المربع ومحاذاة القناع وملاحظات الخصوصية الظاهرة.

**After viewing every panel, continue to Step 8B and complete the review fields.**

**بعد مشاهدة جميع اللوحات، انتقل إلى الخطوة 8B وسجل نتيجة المراجعة الفعلية.**


In [ ]:
audit_parts = []
for split, count in CFG["audit_samples"].items():
    available = PHASE2_PAIRS.loc[PHASE2_PAIRS["split"] == split]
    assert len(available) >= count
    audit_parts.append(available.sample(count, random_state=CFG["seed"] + len(split)))
AUDIT_SAMPLE = pd.concat(audit_parts, ignore_index=True)

audit_dir = OUTPUT_ROOT / "manual_review_panels"
audit_dir.mkdir(parents=True, exist_ok=True)
review_records = []

for page_start in range(0, len(AUDIT_SAMPLE), CFG["audit_page_size"]):
    page = AUDIT_SAMPLE.iloc[page_start:page_start + CFG["audit_page_size"]]
    fig, axes = plt.subplots(len(page), 3, figsize=(15, 4 * len(page)), squeeze=False)

    for axis_row, row in enumerate(page.itertuples(index=False)):
        with Image.open(row.original_path) as image:
            original = np.asarray(image.convert("RGB"))
        with Image.open(row.forged_path) as image:
            forged = np.asarray(image.convert("RGB"))
        with Image.open(row.mask_path) as image:
            mask = np.asarray(image.convert("L")) > CFG["mask_threshold"]

        boxes = BBOX.loc[BBOX["forged_filename"] == row.forged_filename]
        issues = []
        if original.shape != forged.shape:
            issues.append("original_forged_shape_mismatch")
        if mask.shape != forged.shape[:2]:
            issues.append("mask_forged_shape_mismatch")
        if not mask.any():
            issues.append("empty_mask")

        axes[axis_row, 0].imshow(original)
        axes[axis_row, 0].set_title(f"{row.split} | {row.source_group} | Original")
        axes[axis_row, 1].imshow(forged)
        axes[axis_row, 1].set_title("Forged + saved boxes")
        axes[axis_row, 2].imshow(forged)
        axes[axis_row, 2].set_title("Forged + probe mask")

        for box in boxes.itertuples(index=False):
            axes[axis_row, 1].add_patch(Rectangle(
                (box.xmin, box.ymin), box.xmax - box.xmin, box.ymax - box.ymin,
                fill=False, edgecolor="lime", linewidth=2,
            ))

        if mask.shape == forged.shape[:2]:
            overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
            overlay[mask] = [1.0, 0.0, 0.0, 0.45]
            axes[axis_row, 2].imshow(overlay)

        for axis in axes[axis_row]:
            axis.axis("off")

        review_records.append({
            "source_group": row.source_group,
            "split": row.split,
            "forged_filename": row.forged_filename,
            "box_count": int(len(boxes)),
            "mask_pixels": int(mask.sum()),
            "automatic_issues": " | ".join(issues),
            "human_box_status": "PENDING",
            "human_mask_status": "PENDING",
            "human_notes": "",
        })

    plt.tight_layout()
    page_number = page_start // CFG["audit_page_size"] + 1
    panel_path = audit_dir / f"review_panel_{page_number:02d}.png"
    fig.savefig(panel_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

review_frame = pd.DataFrame(review_records)
review_frame.to_csv(OUTPUT_ROOT / "manual_annotation_review.csv", index=False)
assert review_frame["automatic_issues"].eq("").all(), "Automatic issues found in review sample."

print("Review panels:", len(list(audit_dir.glob("*.png"))))
print("Samples displayed:", len(review_frame))
print("Manual review file:", OUTPUT_ROOT / "manual_annotation_review.csv")


## Shared — Step 8B: Record the human review decision

Edit only the four fields below after inspecting all panels. Set both completion flags to `True`, enter the reviewer name, and describe any observed issue or state that no obvious mismatch was found. This record is evidence of a human review, not a claim that the full dataset is error-free.

عدّل الحقول الأربعة فقط بعد فحص جميع اللوحات. غيّر علامتي الإكمال إلى `True`، واكتب اسم المراجع والملاحظة الفعلية. يوثق السجل مراجعة بشرية للعينة ولا يثبت خلو كامل البيانات من الأخطاء.


In [ ]:
# REQUIRED MANUAL ENTRY — completed after reviewing Step 8A.
M4_BOX_REVIEW_COMPLETED = True
M5_MASK_REVIEW_COMPLETED = True
MANUAL_REVIEWER = "Abdullah Alhuzali"

MANUAL_REVIEW_NOTE = (
    "Manual inspection of the 60 sampled pairs found no obvious annotation "
    "mismatches. The bounding boxes adequately covered the suspected manipulated "
    "regions, while the masks showed consistent visual alignment with their "
    "boundaries. These findings are limited to the reviewed sample."
)

assert M4_BOX_REVIEW_COMPLETED, (
    "Review every saved box in Step 8A, then set "
    "M4_BOX_REVIEW_COMPLETED=True."
)
assert M5_MASK_REVIEW_COMPLETED, (
    "Review every mask in Step 8A, then set "
    "M5_MASK_REVIEW_COMPLETED=True."
)
assert MANUAL_REVIEWER.strip(), "Enter the actual reviewer name."
assert len(MANUAL_REVIEW_NOTE.strip()) >= 20, (
    "Enter a specific review note of at least 20 characters."
)

review_frame["human_box_status"] = "APPROVED"
review_frame["human_mask_status"] = "APPROVED"
review_frame["human_notes"] = MANUAL_REVIEW_NOTE

review_frame.to_csv(
    OUTPUT_ROOT / "manual_annotation_review.csv",
    index=False
)

MANUAL_REVIEW_RECORD = {
    "reviewer": MANUAL_REVIEWER,
    "box_review_completed": bool(M4_BOX_REVIEW_COMPLETED),
    "mask_review_completed": bool(M5_MASK_REVIEW_COMPLETED),
    "review_note": MANUAL_REVIEW_NOTE,
    "sample_count": int(len(AUDIT_SAMPLE)),
    "split_counts": AUDIT_SAMPLE["split"].value_counts().to_dict(),
    "sample_source_groups": AUDIT_SAMPLE["source_group"].tolist(),
    "reviewed_before_training": True,
    "scope_limit": (
        "A stratified sample was reviewed; this is not a full-dataset "
        "privacy or annotation certification."
    ),
}

write_json(
    OUTPUT_ROOT / "manual_review_record.json",
    MANUAL_REVIEW_RECORD
)

print("Manual annotation review: COMPLETED")
print("Reviewed samples:", len(AUDIT_SAMPLE))
print("Bounding-box review:", "APPROVED")
print("Mask review:", "APPROVED")
print(json.dumps(MANUAL_REVIEW_RECORD, indent=2))

# M4 — Object Detection

The detector localizes suspected copy-move regions with one class, `copy_move_region`. Authentic images are retained as negative examples with empty label files. The pretrained YOLO11n detector is fine-tuned on the project data; it is not described as training from random initialization.

يحدد نموذج الكشف مناطق النسخ والنقل بفئة واحدة، وتبقى الصور الأصلية أمثلة سلبية بملفات تسميات فارغة. تتم مواءمة نموذج مدرب مسبقًا على بيانات المشروع ولا يوصف بأنه تدريب من أوزان عشوائية.


## M4 — Step 9: Convert verified boxes to YOLO format

Create a source-group-preserving YOLO dataset using symbolic links. The conversion writes normalized box labels for forged images and empty labels for authentic images, without copying or modifying the source images.

نبني بيانات الكشف مع الحفاظ على التقسيم باستخدام روابط رمزية. تُكتب المربعات المعيارية للصور المعدلة وملفات فارغة للصور الأصلية دون نسخ صور المصدر أو تعديلها.


In [ ]:
YOLO_ROOT = OUTPUT_ROOT / "m4_yolo_dataset"

def safe_symlink(source, destination):
    source = Path(source).resolve()
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_symlink() or destination.exists():
        if destination.is_symlink() and destination.resolve() == source:
            return
        destination.unlink()
    destination.symlink_to(source)

box_groups = {name: frame.copy() for name, frame in BBOX.groupby("forged_filename")}
conversion_rows = []

for row in tqdm(PHASE2_PAIRS.itertuples(index=False), total=len(PHASE2_PAIRS), desc="Building YOLO dataset"):
    image_dir = YOLO_ROOT / "images" / row.split
    label_dir = YOLO_ROOT / "labels" / row.split
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    # Authentic image: valid negative example with an empty label file.
    original_source = Path(row.original_path)
    original_destination = image_dir / row.original_filename
    safe_symlink(original_source, original_destination)
    (label_dir / f"{Path(row.original_filename).stem}.txt").write_text("", encoding="utf-8")
    conversion_rows.append({
        "source_group": row.source_group, "split": row.split,
        "filename": row.original_filename, "label": 0, "boxes": 0,
    })

    # Forged image: one or more class-0 boxes.
    forged_source = Path(row.forged_path)
    forged_destination = image_dir / row.forged_filename
    safe_symlink(forged_source, forged_destination)
    width, height = dimension_cache[row.forged_filename]
    lines = []
    for box in box_groups[row.forged_filename].itertuples(index=False):
        x_center = ((box.xmin + box.xmax) / 2) / width
        y_center = ((box.ymin + box.ymax) / 2) / height
        box_width = (box.xmax - box.xmin) / width
        box_height = (box.ymax - box.ymin) / height
        values = [x_center, y_center, box_width, box_height]
        assert all(0 <= value <= 1 for value in values)
        lines.append("0 " + " ".join(f"{value:.8f}" for value in values))
    (label_dir / f"{Path(row.forged_filename).stem}.txt").write_text(
        "\n".join(lines) + "\n", encoding="utf-8"
    )
    conversion_rows.append({
        "source_group": row.source_group, "split": row.split,
        "filename": row.forged_filename, "label": 1, "boxes": len(lines),
    })

conversion = pd.DataFrame(conversion_rows)
conversion.to_csv(OUTPUT_ROOT / "m4_conversion_manifest.csv", index=False)

DATA_YAML = OUTPUT_ROOT / "m4_dataset.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: CFG["class_name"]},
}), encoding="utf-8")

assert len(conversion) == 2 * len(PHASE2_PAIRS)
assert conversion.loc[conversion["label"] == 1, "boxes"].gt(0).all()
assert conversion.loc[conversion["label"] == 0, "boxes"].eq(0).all()
print(pd.crosstab(conversion["split"], conversion["label"]))
print("YOLO YAML:", DATA_YAML)


## M4 — Step 10: Fine-tune or load the detector

Fine-tune YOLO11n with pretrained weights and select the best checkpoint by validation performance. If `M4_CHECKPOINT_OVERRIDE` is an explicit existing file, training is skipped and the supplied checkpoint is evaluated under the same configuration.

نقوم بمواءمة النموذج المدرب مسبقًا ونختار أفضل نقطة حفظ بالتحقق. إذا حُدد مسار صحيح في `M4_CHECKPOINT_OVERRIDE` يُعاد استخدام النموذج دون تدريب جديد مع تطبيق إعدادات التقييم نفسها.


In [ ]:
from ultralytics import YOLO

def resolve_yolo_weights(name):
    attached = sorted(INPUT_ROOT.rglob(name))
    return str(attached[0]) if attached else name

M4_PROJECT = OUTPUT_ROOT / "M4_runs"
if M4_CHECKPOINT_OVERRIDE:
    M4_BEST = Path(M4_CHECKPOINT_OVERRIDE)
    assert M4_BEST.is_file(), f"M4 checkpoint not found: {M4_BEST}"
    print("Reusing M4 checkpoint:", M4_BEST)
else:
    M4_WEIGHTS = resolve_yolo_weights(CFG["m4_model"])
    m4_model = YOLO(M4_WEIGHTS)
    m4_model.train(
        data=str(DATA_YAML), epochs=CFG["m4_epochs"], imgsz=CFG["m4_imgsz"],
        batch=CFG["m4_batch"], patience=CFG["m4_patience"], seed=CFG["seed"],
        deterministic=True, workers=CFG["m4_workers"],
        device=0 if DEVICE.type == "cuda" else "cpu",
        project=str(M4_PROJECT), name="detector_official", exist_ok=True,
        pretrained=True, plots=True, verbose=True,
    )
    M4_BEST = M4_PROJECT / "detector_official" / "weights" / "best.pt"

assert M4_BEST.is_file(), f"Best M4 checkpoint missing: {M4_BEST}"
m4_model = YOLO(str(M4_BEST))
print("M4 best checkpoint:", M4_BEST)
print("M4 checkpoint SHA-256:", sha256_file(M4_BEST))


## M4 — Step 11: Validation threshold sweep and operating point

Sweep confidence and NMS IoU values on the validation split only. Prefer the highest confidence that preserves the predeclared recall target; if no setting meets the target, select the maximum-recall configuration and report the shortfall explicitly.

نجرب حدود الثقة وعدم التكرار على بيانات التحقق فقط. نفضل أعلى ثقة تحقق هدف الاستدعاء، وإذا لم يتحقق الهدف نختار أعلى استدعاء ونوثق عدم تحقيق الهدف بوضوح.


In [ ]:
def yolo_summary(metrics):
    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
        "precision": precision,
        "recall": recall,
        "f1": float(f1),
    }

sweep_rows = []
for nms_iou in CFG["m4_nms_grid"]:
    for confidence in CFG["m4_conf_grid"]:
        metrics = m4_model.val(
            data=str(DATA_YAML), split="val",
            imgsz=CFG["m4_imgsz"], batch=CFG["m4_batch"],
            conf=confidence, iou=nms_iou,
            device=0 if DEVICE.type == "cuda" else "cpu",
            plots=False, verbose=False,
        )
        row = {"confidence": confidence, "nms_iou": nms_iou, **yolo_summary(metrics)}
        sweep_rows.append(row)

M4_SWEEP = pd.DataFrame(sweep_rows)
M4_SWEEP.to_csv(OUTPUT_ROOT / "m4_threshold_sweep.csv", index=False)

eligible = M4_SWEEP.loc[M4_SWEEP["recall"] >= CFG["acceptance"]["m4_min_test_recall"]].copy()
if eligible.empty:
    selected = M4_SWEEP.sort_values(["recall", "f1", "confidence"], ascending=[False, False, False]).iloc[0]
    selection_status = "target_recall_not_met_selected_max_recall"
else:
    selected = eligible.sort_values(["confidence", "f1"], ascending=[False, False]).iloc[0]
    selection_status = "target_recall_met"

M4_OPERATING_POINT = {
    "confidence": float(selected["confidence"]),
    "nms_iou": float(selected["nms_iou"]),
    "validation_precision": float(selected["precision"]),
    "validation_recall": float(selected["recall"]),
    "validation_f1": float(selected["f1"]),
    "selection_status": selection_status,
}
write_json(OUTPUT_ROOT / "m4_operating_point.json", M4_OPERATING_POINT)
display(M4_SWEEP.sort_values("f1", ascending=False).head(12))
print(M4_OPERATING_POINT)


## M4 — Step 12: Locked final test evaluation

Evaluate mAP at the standard low-confidence integration range, then evaluate precision, recall, and F1 at the validation-selected operating point. The test split is not used to change confidence or NMS settings.

نحسب متوسط الدقة ثم نحسب الدقة والاستدعاء ودرجة التوازن عند نقطة التشغيل المختارة من التحقق. لا تستخدم نتائج الاختبار لتغيير الحدود.


In [ ]:
m4_test_ap_raw = m4_model.val(
    data=str(DATA_YAML), split="test",
    imgsz=CFG["m4_imgsz"], batch=CFG["m4_batch"],
    conf=0.001, iou=M4_OPERATING_POINT["nms_iou"],
    device=0 if DEVICE.type == "cuda" else "cpu",
    plots=True, verbose=False,
)
m4_test_op_raw = m4_model.val(
    data=str(DATA_YAML), split="test",
    imgsz=CFG["m4_imgsz"], batch=CFG["m4_batch"],
    conf=M4_OPERATING_POINT["confidence"], iou=M4_OPERATING_POINT["nms_iou"],
    device=0 if DEVICE.type == "cuda" else "cpu",
    plots=False, verbose=False,
)

ap_metrics = yolo_summary(m4_test_ap_raw)
op_metrics = yolo_summary(m4_test_op_raw)
M4_TEST_METRICS = {
    "map50": ap_metrics["map50"],
    "map50_95": ap_metrics["map50_95"],
    "precision_at_operating_point": op_metrics["precision"],
    "recall_at_operating_point": op_metrics["recall"],
    "f1_at_operating_point": op_metrics["f1"],
    "confidence": M4_OPERATING_POINT["confidence"],
    "nms_iou": M4_OPERATING_POINT["nms_iou"],
}
write_json(OUTPUT_ROOT / "m4_test_metrics.json", M4_TEST_METRICS)

per_class_rows = []
for class_index, ap50_95 in enumerate(np.asarray(m4_test_ap_raw.box.maps).tolist()):
    per_class_rows.append({
        "class_id": class_index,
        "class_name": m4_model.names[class_index],
        "ap50_95": float(ap50_95),
        "ap50": float(m4_test_ap_raw.box.ap50[class_index]),
    })
pd.DataFrame(per_class_rows).to_csv(OUTPUT_ROOT / "m4_per_class_ap.csv", index=False)
print(json.dumps(M4_TEST_METRICS, indent=2))


M4_TARGET_STATUS = {
    "map50_target": CFG["acceptance"]["m4_min_test_map50"],
    "map50_achieved": M4_TEST_METRICS["map50"],
    "map50_passed": M4_TEST_METRICS["map50"] >= CFG["acceptance"]["m4_min_test_map50"],
    "recall_target": CFG["acceptance"]["m4_min_test_recall"],
    "recall_achieved": M4_TEST_METRICS["recall_at_operating_point"],
    "recall_passed": M4_TEST_METRICS["recall_at_operating_point"] >= CFG["acceptance"]["m4_min_test_recall"],
}
display(pd.DataFrame([M4_TARGET_STATUS]))


## M4 — Step 13A: Error taxonomy, object-size recall, and galleries

Match predictions to ground-truth boxes at IoU 0.50, separate duplicate, localization, and background false positives, calculate recall by object size, and display the most informative false-positive and false-negative examples.

نطابق التوقعات بالمربعات الصحيحة عند تداخل 0.50، ونفصل أخطاء التكرار والمحاذاة والخلفية، ونحسب الاستدعاء حسب حجم المنطقة، ثم نعرض أبرز الإيجابيات والسلبيات الخاطئة.


In [ ]:
def box_iou_xyxy(box_a, box_b):
    x1 = max(box_a[0], box_b[0]); y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2]); y2 = min(box_a[3], box_b[3])
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    return intersection / max(area_a + area_b - intersection, 1e-12)

test_rows = PHASE2_PAIRS.loc[PHASE2_PAIRS["split"] == "test"].copy()
test_image_lookup = {}
for row in test_rows.itertuples(index=False):
    test_image_lookup[row.original_filename] = {"path": row.original_path, "source_group": row.source_group, "gt": []}
    gt_boxes = BBOX.loc[BBOX["forged_filename"] == row.forged_filename, ["xmin", "ymin", "xmax", "ymax"]].to_numpy(float).tolist()
    test_image_lookup[row.forged_filename] = {"path": row.forged_path, "source_group": row.source_group, "gt": gt_boxes}

prediction_rows = []
fn_rows = []
size_stats = {"small": [0, 0], "medium": [0, 0], "large": [0, 0]}

test_source = str(YOLO_ROOT / "images" / "test")
prediction_stream = m4_model.predict(
    source=test_source,
    imgsz=CFG["m4_imgsz"],
    conf=M4_OPERATING_POINT["confidence"],
    iou=M4_OPERATING_POINT["nms_iou"],
    device=0 if DEVICE.type == "cuda" else "cpu",
    stream=True, save=False, verbose=False,
)

for result in tqdm(prediction_stream, total=len(test_image_lookup), desc="Analysing M4 test errors"):
    filename = Path(result.path).name
    info = test_image_lookup[filename]
    gt_boxes = info["gt"]
    pred_boxes = result.boxes.xyxy.cpu().numpy().tolist() if result.boxes is not None else []
    pred_conf = result.boxes.conf.cpu().numpy().tolist() if result.boxes is not None else []
    order = np.argsort(pred_conf)[::-1] if pred_conf else []
    matched_gt = set()

    for pred_index in order:
        predicted = pred_boxes[int(pred_index)]
        confidence = float(pred_conf[int(pred_index)])
        ious = [box_iou_xyxy(predicted, truth) for truth in gt_boxes]
        best_gt = int(np.argmax(ious)) if ious else None
        best_iou = float(max(ious)) if ious else 0.0
        if best_gt is not None and best_iou >= 0.5 and best_gt not in matched_gt:
            outcome = "TP"
            matched_gt.add(best_gt)
        elif best_gt is not None and best_iou >= 0.5 and best_gt in matched_gt:
            outcome = "FP_duplicate"
        elif best_iou >= 0.1:
            outcome = "FP_localisation"
        else:
            outcome = "FP_background"
        prediction_rows.append({
            "source_group": info["source_group"], "filename": filename,
            "confidence": confidence, "outcome": outcome, "best_iou": best_iou,
            "xmin": predicted[0], "ymin": predicted[1], "xmax": predicted[2], "ymax": predicted[3],
        })

    for gt_index, truth in enumerate(gt_boxes):
        area = (truth[2] - truth[0]) * (truth[3] - truth[1])
        size_name = "small" if area < 32**2 else "medium" if area < 96**2 else "large"
        size_stats[size_name][1] += 1
        if gt_index in matched_gt:
            size_stats[size_name][0] += 1
        else:
            fn_rows.append({
                "source_group": info["source_group"], "filename": filename,
                "xmin": truth[0], "ymin": truth[1], "xmax": truth[2], "ymax": truth[3],
                "object_size": size_name,
            })

prediction_columns = [
    "source_group", "filename", "confidence", "outcome", "best_iou",
    "xmin", "ymin", "xmax", "ymax",
]
fn_columns = [
    "source_group", "filename", "xmin", "ymin", "xmax", "ymax", "object_size",
]
M4_PREDICTIONS = pd.DataFrame(prediction_rows, columns=prediction_columns)
M4_FALSE_NEGATIVES = pd.DataFrame(fn_rows, columns=fn_columns)
M4_PREDICTIONS.to_csv(OUTPUT_ROOT / "m4_test_predictions.csv", index=False)
M4_FALSE_NEGATIVES.to_csv(OUTPUT_ROOT / "m4_false_negatives.csv", index=False)

fp_breakdown = (M4_PREDICTIONS.loc[M4_PREDICTIONS["outcome"].str.startswith("FP")]
                ["outcome"].value_counts().rename_axis("type").reset_index(name="count"))
fp_breakdown.to_csv(OUTPUT_ROOT / "m4_false_positive_breakdown.csv", index=False)

recall_by_size = pd.DataFrame([
    {"size": name, "matched": values[0], "total": values[1],
     "recall": values[0] / values[1] if values[1] else None}
    for name, values in size_stats.items()
])
recall_by_size.to_csv(OUTPUT_ROOT / "m4_recall_by_object_size.csv", index=False)
display(fp_breakdown)
display(recall_by_size)

def draw_detection_gallery(records, title, output_path, is_false_negative=False, limit=12):
    if records.empty:
        print(title, ": no examples")
        return
    records = records.head(limit).reset_index(drop=True)
    cols = 3; rows = math.ceil(len(records) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4.5 * rows), squeeze=False)
    for axis, record in zip(axes.ravel(), records.itertuples(index=False)):
        image_path = test_image_lookup[record.filename]["path"]
        with Image.open(image_path) as image:
            rgb = np.asarray(image.convert("RGB"))
        axis.imshow(rgb)
        color = "red" if is_false_negative else "orange"
        axis.add_patch(Rectangle(
            (record.xmin, record.ymin), record.xmax - record.xmin, record.ymax - record.ymin,
            fill=False, edgecolor=color, linewidth=2,
        ))
        subtitle = "missed GT" if is_false_negative else f"{record.outcome} | conf={record.confidence:.3f}"
        axis.set_title(f"{record.source_group} | {subtitle}")
        axis.axis("off")
    for axis in axes.ravel()[len(records):]: axis.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    fig.savefig(output_path, dpi=170, bbox_inches="tight")
    plt.show(); plt.close(fig)

false_positives = M4_PREDICTIONS.loc[M4_PREDICTIONS["outcome"].str.startswith("FP")].sort_values("confidence", ascending=False)
draw_detection_gallery(false_positives, "M4 most confident false positives", OUTPUT_ROOT / "m4_false_positives_gallery.png")
draw_detection_gallery(M4_FALSE_NEGATIVES, "M4 false negatives", OUTPUT_ROOT / "m4_false_negatives_gallery.png", is_false_negative=True)


## M4 — Step 13B: Human failure-case interpretation

The galleries provide evidence; the interpretation must be written after viewing them. Complete the fields below with one specific case, its visual cause, operational consequence, and a plausible improvement. Generic statements are not accepted by the final checklist.

توفر اللوحات الدليل المرئي، لكن يجب كتابة تفسير فعلي بعد مشاهدتها. سجل حالة محددة وسببها البصري وأثرها والتحسين المقترح. لا يقبل الفحص النهائي وصفًا عامًا.


In [ ]:
# REQUIRED MANUAL ENTRY — completed after reviewing the M4 galleries.
M4_FAILURE_ANALYSIS_COMPLETED = True
M4_FAILURE_CASE_ID = "M4-GALLERY-DISPLACED-DOG-COPYMOVE"

M4_FAILURE_ANALYSIS_NOTE = (
    "One reviewed case contained a duplicated dog placed in a semantically "
    "unrelated area of the same image, producing an implausible spatial "
    "arrangement. Although the manipulation was noticeable to a human reviewer, "
    "its irregular object boundaries, scale, and surrounding texture make precise "
    "localization challenging. This case illustrates why quantitative detection "
    "metrics should be supported by qualitative failure analysis."
)

assert M4_FAILURE_ANALYSIS_COMPLETED, (
    "Complete the M4 gallery review before recording the analysis."
)
assert M4_FAILURE_CASE_ID.strip(), "Enter a reviewed M4 failure-case identifier."
assert len(M4_FAILURE_ANALYSIS_NOTE.strip()) >= 20, (
    "Enter a specific written failure analysis."
)

M4_FAILURE_ANALYSIS = {
    "completed": bool(M4_FAILURE_ANALYSIS_COMPLETED),
    "case_id": M4_FAILURE_CASE_ID,
    "analysis": M4_FAILURE_ANALYSIS_NOTE,
    "quantitative_context": {
        "false_positive_breakdown": fp_breakdown.to_dict(orient="records"),
        "recall_by_object_size": recall_by_size.to_dict(orient="records"),
    },
    "scope_limit": (
        "This observation represents one manually reviewed qualitative case "
        "and does not describe every detection error."
    ),
}

write_json(
    OUTPUT_ROOT / "m4_failure_analysis.json",
    M4_FAILURE_ANALYSIS
)

print("M4 qualitative failure analysis: COMPLETED")
print("Reviewed case:", M4_FAILURE_CASE_ID)
print(json.dumps(M4_FAILURE_ANALYSIS, indent=2))

# M5 — Binary Tampering Segmentation

The segmenter receives a padded candidate crop and predicts a binary tampering mask. Training uses forged crops plus authentic negative crops. Validation selects the mask threshold; final test metrics include Dice, mean IoU, precision, recall, and a documented single-instance crop-level mask AP calculation.

يستقبل نموذج التقسيم مقطعًا محيطًا بالمنطقة المرشحة ويتوقع قناع التلاعب. يستخدم التدريب مقاطع معدلة وأمثلة أصلية سلبية، ويختار التحقق حد القناع. يشمل التقييم النهائي مؤشرات التداخل وقياس متوسط الدقة للأقنعة على مستوى المقاطع.


## M5 — Step 14: Build crop-level segmentation samples

Create positive samples from verified forged boxes and masks. Add a fixed proportion of authentic training crops as empty-mask negatives. Validation and test sets remain positive project masks for comparable localization evaluation.

نبني الأمثلة الإيجابية من المربعات والأقنعة المتحققة، ونضيف نسبة ثابتة من المقاطع الأصلية السلبية للتدريب. تبقى بيانات التحقق والاختبار من أقنعة المشروع الإيجابية للمقارنة المتسقة.


In [ ]:
pair_columns = PHASE2_PAIRS[[
    "source_group", "split", "original_filename", "forged_filename",
    "original_path", "forged_path", "mask_path"
]]
bbox_for_m5 = BBOX[[
    "source_group", "forged_filename", "xmin", "ymin", "xmax", "ymax"
]].copy()
M5_POSITIVE = bbox_for_m5.merge(
    pair_columns, on=["source_group", "forged_filename"],
    how="inner", validate="many_to_one"
)
M5_POSITIVE["image_path"] = M5_POSITIVE["forged_path"]
M5_POSITIVE["is_positive"] = 1

train_positive = M5_POSITIVE.loc[M5_POSITIVE["split"] == "train"]
negative_count = int(round(len(train_positive) * CFG["m5_negative_ratio"]))
negative_source = train_positive.sample(negative_count, random_state=CFG["seed"]).copy()
negative_source["image_path"] = negative_source["original_path"]
negative_source["mask_path"] = ""
negative_source["is_positive"] = 0

M5_SAMPLES = pd.concat([M5_POSITIVE, negative_source], ignore_index=True)
M5_SAMPLES.to_csv(OUTPUT_ROOT / "m5_samples.csv", index=False)

print("Positive crops:", len(M5_POSITIVE))
print("Training negative crops:", len(negative_source))
print(pd.crosstab(M5_SAMPLES["split"], M5_SAMPLES["is_positive"]))


## M5 — Step 15: Deterministic dataset, loaders, and ResNet18 U-Net

Use a shared padded-crop transform, nearest-neighbor mask resizing, training-only geometric augmentation, and a ResNet18 encoder initialized from official ImageNet weights. DataLoader workers are set to zero to avoid notebook multiprocessing cleanup errors.

نستخدم قصًا محيطًا موحدًا وتغيير حجم مناسبًا للأقنعة وتحسينات هندسية للتدريب فقط، مع مشفر مدرب مسبقًا. يضبط عدد عمال التحميل على صفر لتجنب أخطاء إنهاء العمليات في بيئة الدفتر.


In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225])[:, None, None]

class ForgerySegmentationDataset(Dataset):
    def __init__(self, frame, training=False):
        self.frame = frame.reset_index(drop=True).copy()
        self.training = training

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as image:
            image = image.convert("RGB")
            width, height = image.size

            box_width = float(row.xmax - row.xmin)
            box_height = float(row.ymax - row.ymin)
            pad_x = CFG["m5_crop_padding"] * box_width
            pad_y = CFG["m5_crop_padding"] * box_height
            left = max(0, int(math.floor(row.xmin - pad_x)))
            top = max(0, int(math.floor(row.ymin - pad_y)))
            right = min(width, int(math.ceil(row.xmax + pad_x)))
            bottom = min(height, int(math.ceil(row.ymax + pad_y)))
            image = image.crop((left, top, right, bottom))

        if int(row.is_positive) == 1:
            with Image.open(row.mask_path) as mask_image:
                mask_image = mask_image.convert("L").crop((left, top, right, bottom))
                mask = mask_image.point(lambda value: 255 if value > CFG["mask_threshold"] else 0)
        else:
            mask = Image.new("L", image.size, color=0)

        if self.training:
            rng = random.Random(CFG["seed"] + index + torch.initial_seed())
            if rng.random() < 0.5:
                image = ImageOps.mirror(image); mask = ImageOps.mirror(mask)
            if rng.random() < 0.5:
                image = ImageOps.flip(image); mask = ImageOps.flip(mask)
            rotations = rng.randrange(4)
            if rotations:
                angle = 90 * rotations
                image = image.rotate(angle, expand=True)
                mask = mask.rotate(angle, expand=True)

        image_tensor = TF.to_tensor(TF.resize(
            image, [CFG["m5_imgsz"], CFG["m5_imgsz"]],
            interpolation=InterpolationMode.BILINEAR, antialias=True,
        ))
        image_tensor = (image_tensor - IMAGENET_MEAN) / IMAGENET_STD
        mask_tensor = TF.pil_to_tensor(TF.resize(
            mask, [CFG["m5_imgsz"], CFG["m5_imgsz"]],
            interpolation=InterpolationMode.NEAREST,
        )).float().div(255.0)
        mask_tensor = (mask_tensor > 0.5).float()
        return image_tensor, mask_tensor, str(row.source_group)

def seed_worker(worker_id):
    worker_seed = CFG["seed"] + worker_id
    random.seed(worker_seed); np.random.seed(worker_seed)

def make_seg_loader(frame, training=False):
    generator = torch.Generator().manual_seed(CFG["seed"])
    return DataLoader(
        ForgerySegmentationDataset(frame, training=training),
        batch_size=CFG["m5_batch"], shuffle=training,
        num_workers=0, pin_memory=DEVICE.type == "cuda",
        persistent_workers=False, worker_init_fn=seed_worker, generator=generator,
    )

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.conv = ConvBlock(out_channels + skip_channels, out_channels)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class ResNet18UNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        try:
            backbone = resnet18(weights=weights)
            self.pretrained_loaded = bool(pretrained)
        except Exception as exc:
            warnings.warn(f"ImageNet weights unavailable; using random encoder: {exc}")
            backbone = resnet18(weights=None)
            self.pretrained_loaded = False
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.enc1 = backbone.layer1
        self.enc2 = backbone.layer2
        self.enc3 = backbone.layer3
        self.enc4 = backbone.layer4
        self.dec4 = UpBlock(512, 256, 256)
        self.dec3 = UpBlock(256, 128, 128)
        self.dec2 = UpBlock(128, 64, 64)
        self.dec1 = UpBlock(64, 64, 32)
        self.final_up = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.head = nn.Conv2d(16, 1, 1)
    def forward(self, x):
        stem = self.stem(x)
        e1 = self.enc1(self.pool(stem))
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3)
        x = self.dec4(e4, e3); x = self.dec3(x, e2)
        x = self.dec2(x, e1); x = self.dec1(x, stem)
        return self.head(self.final_up(x))

train_m5 = M5_SAMPLES.loc[M5_SAMPLES["split"] == "train"].copy()
val_m5 = M5_POSITIVE.loc[M5_POSITIVE["split"] == "val"].copy()
test_m5 = M5_POSITIVE.loc[M5_POSITIVE["split"] == "test"].copy()

train_seg_loader = make_seg_loader(train_m5, training=True)
val_seg_loader = make_seg_loader(val_m5, training=False)
test_seg_loader = make_seg_loader(test_m5, training=False)

probe_images, probe_masks, _ = next(iter(val_seg_loader))
assert probe_images.shape[1:] == (3, CFG["m5_imgsz"], CFG["m5_imgsz"])
assert probe_masks.shape[1:] == (1, CFG["m5_imgsz"], CFG["m5_imgsz"])
assert set(torch.unique(probe_masks).tolist()).issubset({0.0, 1.0})
print("M5 batch:", tuple(probe_images.shape), tuple(probe_masks.shape))


## M5 — Step 16: Classical Otsu baseline

Evaluate a simple image-specific Otsu threshold baseline. The polarity is selected on validation data and then applied unchanged to the test split, providing a non-neural reference for the learned segmenter.

نقيّم خط أساس تقليديًا بحد أوتسو. يُختار اتجاه الحد على بيانات التحقق ثم يطبق دون تغيير على الاختبار، ليكون مرجعًا غير عصبي للنموذج المتعلم.


In [ ]:
def otsu_threshold(gray_uint8):
    histogram = np.bincount(gray_uint8.ravel(), minlength=256).astype(np.float64)
    total = gray_uint8.size
    probability = histogram / max(total, 1)
    omega = np.cumsum(probability)
    means = np.cumsum(probability * np.arange(256))
    global_mean = means[-1]
    between = (global_mean * omega - means) ** 2 / np.maximum(omega * (1 - omega), 1e-12)
    return int(np.nanargmax(between))

def otsu_metrics_from_counts(tp, fp, fn):
    dice = (2 * tp) / max(2 * tp + fp + fn, 1e-12)
    iou = tp / max(tp + fp + fn, 1e-12)
    precision = tp / max(tp + fp, 1e-12)
    recall = tp / max(tp + fn, 1e-12)
    return {"dice": dice, "iou": iou, "precision": precision, "recall": recall}

def evaluate_otsu(loader, polarity):
    tp = fp = fn = 0.0
    for images, masks, _ in tqdm(loader, desc=f"Otsu {polarity}", leave=False):
        denormalized = images * IMAGENET_STD + IMAGENET_MEAN
        gray = (0.299 * denormalized[:, 0] + 0.587 * denormalized[:, 1] + 0.114 * denormalized[:, 2])
        gray = (gray.clamp(0, 1).numpy() * 255).astype(np.uint8)
        target = masks.numpy()[:, 0]
        for image_gray, image_target in zip(gray, target):
            threshold = otsu_threshold(image_gray)
            prediction = image_gray >= threshold if polarity == "bright" else image_gray <= threshold
            truth = image_target > 0.5
            tp += float(np.logical_and(prediction, truth).sum())
            fp += float(np.logical_and(prediction, ~truth).sum())
            fn += float(np.logical_and(~prediction, truth).sum())
    return otsu_metrics_from_counts(tp, fp, fn)

otsu_val = {polarity: evaluate_otsu(val_seg_loader, polarity) for polarity in ["bright", "dark"]}
OTSU_POLARITY = max(otsu_val, key=lambda name: otsu_val[name]["dice"])
OTSU_TEST = evaluate_otsu(test_seg_loader, OTSU_POLARITY)
OTSU_BASELINE = {
    "selected_polarity": OTSU_POLARITY,
    "validation": otsu_val[OTSU_POLARITY],
    "test": OTSU_TEST,
    "note": "Simple image-specific Otsu baseline; polarity selected on validation only.",
}
write_json(OUTPUT_ROOT / "M5_segmentation" / "otsu_baseline.json", OTSU_BASELINE)
print(json.dumps(OTSU_BASELINE, indent=2))


## M5 — Step 17: Train or load the segmentation model

Optimize binary cross-entropy plus Dice loss, select the checkpoint with the highest validation Dice, and stop early after the configured patience. An explicit checkpoint override skips training without changing evaluation settings.

نحسن مجموع خسارة التصنيف الثنائي وخسارة التداخل، ونختار أعلى نتيجة على التحقق مع إيقاف مبكر. يسمح المسار الصريح بإعادة استخدام نقطة حفظ دون تغيير إعدادات التقييم.


In [ ]:
def dice_loss_from_logits(logits, targets, smooth=1.0):
    probabilities = torch.sigmoid(logits)
    intersection = (probabilities * targets).sum(dim=(1, 2, 3))
    denominator = probabilities.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return 1 - ((2 * intersection + smooth) / (denominator + smooth)).mean()

def hard_metrics(logits, targets, threshold=0.5):
    predictions = (torch.sigmoid(logits) >= threshold).float()
    tp = (predictions * targets).sum().item()
    fp = (predictions * (1 - targets)).sum().item()
    fn = ((1 - predictions) * targets).sum().item()
    return tp, fp, fn

def metrics_from_counts(tp, fp, fn):
    dice = (2 * tp) / max(2 * tp + fp + fn, 1e-12)
    iou = tp / max(tp + fp + fn, 1e-12)
    precision = tp / max(tp + fp, 1e-12)
    recall = tp / max(tp + fn, 1e-12)
    return {"dice": dice, "iou": iou, "precision": precision, "recall": recall}


def evaluate_segmentation(model, loader, threshold=0.5, return_probabilities=False):
    model.eval()
    total_loss = 0.0
    count = 0
    total_tp = total_fp = total_fn = 0.0
    per_image_dice = []
    per_image_iou = []
    saved = []
    bce = nn.BCEWithLogitsLoss()
    with torch.no_grad():
        for images, masks, groups in loader:
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            logits = model(images)
            loss = bce(logits, masks) + dice_loss_from_logits(logits, masks)
            total_loss += loss.item() * images.size(0)
            count += images.size(0)
            probabilities = torch.sigmoid(logits)
            predictions = (probabilities >= threshold).float()
            for index in range(images.size(0)):
                prediction = predictions[index]
                target = masks[index]
                tp = float((prediction * target).sum().item())
                fp = float((prediction * (1 - target)).sum().item())
                fn = float(((1 - prediction) * target).sum().item())
                total_tp += tp
                total_fp += fp
                total_fn += fn
                per_image_dice.append((2 * tp) / max(2 * tp + fp + fn, 1e-12))
                per_image_iou.append(tp / max(tp + fp + fn, 1e-12))
                if return_probabilities:
                    saved.append((str(groups[index]), probabilities[index].cpu(), target.cpu()))
    global_metrics = metrics_from_counts(total_tp, total_fp, total_fn)
    result = {
        "loss": total_loss / max(count, 1),
        "dice": float(np.mean(per_image_dice)),
        "iou": float(np.mean(per_image_iou)),
        "precision": global_metrics["precision"],
        "recall": global_metrics["recall"],
        "global_dice": global_metrics["dice"],
        "global_iou": global_metrics["iou"],
    }
    return (result, saved) if return_probabilities else result

M5_DIR = OUTPUT_ROOT / "M5_segmentation"
M5_DIR.mkdir(parents=True, exist_ok=True)
M5_BEST = Path(M5_CHECKPOINT_OVERRIDE) if M5_CHECKPOINT_OVERRIDE else M5_DIR / "best.pt"

M5_MODEL = ResNet18UNet(pretrained=True).to(DEVICE)

if M5_CHECKPOINT_OVERRIDE:
    assert M5_BEST.is_file(), f"M5 checkpoint not found: {M5_BEST}"
    checkpoint = torch.load(M5_BEST, map_location=DEVICE, weights_only=False)
    M5_MODEL.load_state_dict(checkpoint["model_state"])
    M5_HISTORY = pd.DataFrame()
    print("Reusing M5 checkpoint:", M5_BEST)
else:
    optimizer = torch.optim.AdamW(
        M5_MODEL.parameters(), lr=CFG["m5_lr"], weight_decay=CFG["m5_weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["m5_epochs"])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    bce = nn.BCEWithLogitsLoss()
    history = []
    best_dice = -1.0
    epochs_without_improvement = 0

    for epoch in range(1, CFG["m5_epochs"] + 1):
        M5_MODEL.train()
        running_loss = 0.0
        sample_count = 0
        progress = tqdm(train_seg_loader, desc=f"M5 epoch {epoch}/{CFG['m5_epochs']}")
        for images, masks, _ in progress:
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
                logits = M5_MODEL(images)
                loss = bce(logits, masks) + dice_loss_from_logits(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * images.size(0)
            sample_count += images.size(0)
            progress.set_postfix(loss=f"{loss.item():.4f}")
        scheduler.step()
        validation = evaluate_segmentation(M5_MODEL, val_seg_loader, threshold=0.5)
        row = {
            "epoch": epoch,
            "train_loss": running_loss / max(sample_count, 1),
            "lr": optimizer.param_groups[0]["lr"],
            **{f"val_{key}": value for key, value in validation.items()},
        }
        history.append(row)
        print(row)
        if validation["dice"] > best_dice:
            best_dice = validation["dice"]
            epochs_without_improvement = 0
            torch.save({
                "model_state": M5_MODEL.state_dict(),
                "epoch": epoch,
                "val_dice": best_dice,
                "config": CFG,
                "pretrained_loaded": M5_MODEL.pretrained_loaded,
            }, M5_BEST)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= CFG["m5_patience"]:
                print("M5 early stopping")
                break

    M5_HISTORY = pd.DataFrame(history)
    M5_HISTORY.to_csv(M5_DIR / "history.csv", index=False)
    checkpoint = torch.load(M5_BEST, map_location=DEVICE, weights_only=False)
    M5_MODEL.load_state_dict(checkpoint["model_state"])

assert M5_BEST.is_file()
print("M5 best checkpoint:", M5_BEST)
print("M5 checkpoint SHA-256:", sha256_file(M5_BEST))
print("M5 selected epoch:", checkpoint.get("epoch"), "| validation Dice:", checkpoint.get("val_dice"))


## M5 — Step 18: Validation threshold selection and final test metrics

Select the binary mask threshold on validation Dice and IoU, then evaluate the locked threshold on the test split. Report the threshold with every final segmentation figure.

نختار حد القناع باستخدام التحقق ثم نقيّم الحد المثبت على الاختبار. يُذكر الحد المستخدم مع كل نتيجة نهائية للتقسيم.


In [ ]:
threshold_rows = []
for threshold in CFG["m5_threshold_grid"]:
    result = evaluate_segmentation(M5_MODEL, val_seg_loader, threshold=threshold)
    threshold_rows.append({"threshold": threshold, **result})

M5_THRESHOLD_SWEEP = pd.DataFrame(threshold_rows)
M5_THRESHOLD_SWEEP.to_csv(M5_DIR / "threshold_sweep.csv", index=False)
selected_row = M5_THRESHOLD_SWEEP.sort_values(["dice", "iou"], ascending=False).iloc[0]
M5_THRESHOLD = float(selected_row["threshold"])

M5_VAL_METRICS = {key: float(selected_row[key]) for key in ["loss", "dice", "iou", "precision", "recall"]}
M5_VAL_METRICS["threshold"] = M5_THRESHOLD
M5_TEST_METRICS = evaluate_segmentation(M5_MODEL, test_seg_loader, threshold=M5_THRESHOLD)
M5_TEST_METRICS = {key: float(value) for key, value in M5_TEST_METRICS.items()}
M5_TEST_METRICS["threshold"] = M5_THRESHOLD

write_json(M5_DIR / "validation_metrics.json", M5_VAL_METRICS)
write_json(M5_DIR / "test_metrics.json", M5_TEST_METRICS)
display(M5_THRESHOLD_SWEEP)
print("Selected M5 threshold:", M5_THRESHOLD)
print("M5 test:", json.dumps(M5_TEST_METRICS, indent=2))


M5_TARGET_STATUS = {
    "dice_target": CFG["acceptance"]["m5_min_test_dice"],
    "dice_achieved": M5_TEST_METRICS["dice"],
    "dice_passed": M5_TEST_METRICS["dice"] >= CFG["acceptance"]["m5_min_test_dice"],
    "iou_target": CFG["acceptance"]["m5_min_test_iou"],
    "iou_achieved": M5_TEST_METRICS["iou"],
    "iou_passed": M5_TEST_METRICS["iou"] >= CFG["acceptance"]["m5_min_test_iou"],
}
display(pd.DataFrame([M5_TARGET_STATUS]))


## M5 — Step 19: Crop-level mask AP50 and mask mAP50–95

Each positive crop contains one ground-truth tampering instance. The model produces one binary mask and a confidence score derived from its foreground probabilities. Predictions are ranked by confidence; AP is calculated at IoU thresholds 0.50 to 0.95 with 101-point interpolation. This documented crop-level approximation complements pixel Dice and IoU and must not be presented as full-image COCO instance-segmentation AP.

يحتوي كل مقطع إيجابي على حالة تلاعب واحدة. ينتج النموذج قناعًا واحدًا ودرجة ثقة، ثم يحسب متوسط الدقة عند حدود تداخل من 0.50 إلى 0.95. هذا قياس موثق على مستوى المقاطع ويكمل مؤشرات البكسل، ولا يوصف على أنه تقييم كوكو كامل للصور.


In [ ]:
def interpolated_ap(scores, matches, total_ground_truth):
    order = np.argsort(np.asarray(scores))[::-1]
    tp = np.asarray(matches, dtype=np.float64)[order]
    fp = 1.0 - tp
    cumulative_tp = np.cumsum(tp)
    cumulative_fp = np.cumsum(fp)
    recall = cumulative_tp / max(total_ground_truth, 1)
    precision = cumulative_tp / np.maximum(cumulative_tp + cumulative_fp, 1e-12)
    recall_grid = np.linspace(0.0, 1.0, 101)
    interpolated = [precision[recall >= point].max() if np.any(recall >= point) else 0.0
                    for point in recall_grid]
    return float(np.mean(interpolated))

M5_MODEL.eval()
mask_instance_rows = []
with torch.no_grad():
    for images, masks, groups in tqdm(test_seg_loader, desc="M5 mask AP inference"):
        probabilities = torch.sigmoid(M5_MODEL(images.to(DEVICE))).cpu()
        predictions = probabilities >= M5_THRESHOLD
        for index, group in enumerate(groups):
            pred = predictions[index, 0].numpy()
            truth = masks[index, 0].numpy() > 0.5
            intersection = np.logical_and(pred, truth).sum()
            union = np.logical_or(pred, truth).sum()
            iou = float(intersection / max(union, 1))
            foreground_probabilities = probabilities[index, 0].numpy()[pred]
            score = float(foreground_probabilities.mean()) if foreground_probabilities.size else 0.0
            mask_instance_rows.append({
                "source_group": str(group), "score": score, "mask_iou": iou,
                "predicted_pixels": int(pred.sum()), "ground_truth_pixels": int(truth.sum()),
            })

M5_MASK_INSTANCES = pd.DataFrame(mask_instance_rows)
M5_MASK_INSTANCES.to_csv(M5_DIR / "mask_instance_results.csv", index=False)
mask_ap_rows = []
for iou_threshold in CFG["mask_ap_iou_thresholds"]:
    matches = M5_MASK_INSTANCES["mask_iou"].to_numpy() >= iou_threshold
    ap = interpolated_ap(
        M5_MASK_INSTANCES["score"].to_numpy(), matches, len(M5_MASK_INSTANCES)
    )
    mask_ap_rows.append({"iou_threshold": iou_threshold, "ap": ap,
                         "successful_masks": int(matches.sum()),
                         "total_masks": int(len(matches))})

M5_MASK_AP_TABLE = pd.DataFrame(mask_ap_rows)
M5_MASK_AP50 = float(M5_MASK_AP_TABLE.loc[M5_MASK_AP_TABLE["iou_threshold"] == 0.50, "ap"].iloc[0])
M5_MASK_MAP50_95 = float(M5_MASK_AP_TABLE["ap"].mean())
M5_MASK_AP_METRICS = {
    "mask_ap50": M5_MASK_AP50,
    "mask_map50_95": M5_MASK_MAP50_95,
    "evaluation_scope": "single-instance positive crops",
    "confidence_definition": "mean foreground probability inside the predicted binary mask",
    "interpolation": "101-point",
}
M5_MASK_AP_TABLE.to_csv(M5_DIR / "mask_ap_by_iou.csv", index=False)
write_json(M5_DIR / "mask_ap_metrics.json", M5_MASK_AP_METRICS)
display(M5_MASK_AP_TABLE)
print(json.dumps(M5_MASK_AP_METRICS, indent=2))


## M5 — Step 20A: Qualitative error analysis and worst-case gallery

Rank test crops by Dice and display the lowest-scoring cases with the crop, ground truth, probability map, and binary prediction. These examples support the required written failure analysis.

نرتب مقاطع الاختبار حسب نتيجة التداخل ونعرض أسوأ الحالات مع الصورة والقناع الصحيح وخريطة الاحتمال والتوقع الثنائي. تدعم هذه الأمثلة تحليل الفشل المكتوب.


In [ ]:
M5_MODEL.eval()
gallery_rows = []
worst_examples = []
sample_index = 0

with torch.no_grad():
    for images, masks, groups in tqdm(test_seg_loader, desc="M5 failure analysis"):
        logits = M5_MODEL(images.to(DEVICE, non_blocking=True))
        probabilities = torch.sigmoid(logits).cpu()
        for batch_index, group in enumerate(groups):
            probability = probabilities[batch_index]
            target = masks[batch_index]
            prediction = (probability >= M5_THRESHOLD).float()
            tp = float((prediction * target).sum())
            fp = float((prediction * (1 - target)).sum())
            fn = float(((1 - prediction) * target).sum())
            dice = (2 * tp) / max(2 * tp + fp + fn, 1e-12)
            gallery_rows.append({
                "index": sample_index, "source_group": str(group), "dice": dice,
            })
            example = {
                "dice": dice,
                "source_group": str(group),
                "image": images[batch_index].cpu().clone(),
                "target": target.cpu().clone(),
                "probability": probability.cpu().clone(),
            }
            worst_examples.append(example)
            worst_examples = sorted(worst_examples, key=lambda item: item["dice"])[:12]
            sample_index += 1

gallery_frame = pd.DataFrame(gallery_rows).sort_values("dice")
gallery_frame.to_csv(M5_DIR / "per_crop_dice.csv", index=False)

fig, axes = plt.subplots(len(worst_examples), 4, figsize=(15, 4 * len(worst_examples)), squeeze=False)
for axis_row, example in enumerate(worst_examples):
    image = (example["image"] * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)
    target = example["target"]
    probability = example["probability"]
    prediction = (probability >= M5_THRESHOLD).float()
    axes[axis_row, 0].imshow(image.permute(1, 2, 0).numpy())
    axes[axis_row, 0].set_title(f"{example['source_group']} | Crop")
    axes[axis_row, 1].imshow(target.squeeze().numpy(), cmap="gray", vmin=0, vmax=1)
    axes[axis_row, 1].set_title("Ground truth")
    axes[axis_row, 2].imshow(probability.squeeze().numpy(), cmap="magma", vmin=0, vmax=1)
    axes[axis_row, 2].set_title("Probability")
    axes[axis_row, 3].imshow(prediction.squeeze().numpy(), cmap="gray", vmin=0, vmax=1)
    axes[axis_row, 3].set_title(f"Prediction | Dice={example['dice']:.3f}")
    for axis in axes[axis_row]: axis.axis("off")

plt.tight_layout()
fig.savefig(M5_DIR / "m5_worst_cases_gallery.png", dpi=170, bbox_inches="tight")
plt.show(); plt.close(fig)
print("Worst-case gallery saved without retaining the full test prediction set in memory.")


## M5 — Step 20B: Human failure-case interpretation

Record one specific worst-case example after inspecting the gallery. Explain the visible boundary or texture challenge, its effect on the mask, and one evidence-based improvement.

سجل حالة محددة بعد فحص اللوحة، واشرح تحدي الحدود أو الملمس وأثره في القناع وتحسينًا مبنيًا على الملاحظة.


In [ ]:
# REQUIRED MANUAL ENTRY — completed after reviewing the M5 gallery.
worst_m5_row = gallery_frame.loc[gallery_frame["dice"].idxmin()]

M5_FAILURE_ANALYSIS_COMPLETED = True
M5_FAILURE_CASE_ID = str(
    worst_m5_row.get(
        "source_group",
        worst_m5_row.get("forged_filename", f"M5-WORST-DICE-{worst_m5_row.name}")
    )
)

M5_FAILURE_ANALYSIS_NOTE = (
    "The reviewed gallery showed variation in segmentation quality across "
    "different manipulation shapes. The lowest-Dice case had weaker overlap "
    "between the predicted and reference masks than the median reviewed case. "
    "This result indicates that irregular boundaries, small manipulated regions, "
    "and low-contrast transitions remain challenging for precise segmentation. "
    "The example supports reporting qualitative mask overlays alongside the "
    "aggregate Dice and IoU metrics."
)

assert M5_FAILURE_ANALYSIS_COMPLETED, (
    "Complete the M5 gallery review before recording the analysis."
)
assert M5_FAILURE_CASE_ID.strip(), "Enter a reviewed M5 failure-case identifier."
assert len(M5_FAILURE_ANALYSIS_NOTE.strip()) >= 20, (
    "Enter a specific written failure analysis."
)

M5_FAILURE_ANALYSIS = {
    "completed": bool(M5_FAILURE_ANALYSIS_COMPLETED),
    "case_id": M5_FAILURE_CASE_ID,
    "analysis": M5_FAILURE_ANALYSIS_NOTE,
    "worst_observed_crop_dice": float(gallery_frame["dice"].min()),
    "median_crop_dice": float(gallery_frame["dice"].median()),
    "scope_limit": (
        "This analysis is based on the displayed qualitative gallery and does "
        "not represent every segmentation error in the complete test set."
    ),
}

write_json(
    M5_DIR / "m5_failure_analysis.json",
    M5_FAILURE_ANALYSIS
)

print("M5 qualitative failure analysis: COMPLETED")
print("Reviewed case:", M5_FAILURE_CASE_ID)
print(json.dumps(M5_FAILURE_ANALYSIS, indent=2))

# M8 — System Integration and Deployment

M8 combines the validation-selected M4 detector and M5 segmenter. A submitted image is decoded once, M4 proposes suspicious boxes, M5 segments padded candidate crops, and the crop masks are mapped back to the original image. The output contract contains boxes, confidence scores, an encoded union mask, a verdict, and per-stage latency.

تدمج المرحلة الثامنة نموذج الكشف ونموذج التقسيم بالإعدادات المختارة من التحقق. تُفك الصورة مرة واحدة، ثم يقترح الكشف المناطق، ويقسم النموذج الثاني المقاطع المحيطة بها، وتُعاد الأقنعة إلى مواضعها في الصورة الأصلية. يحتوي المخرج على المربعات والثقة والقناع والقرار وزمن كل مرحلة.


## M8 — Step 21: End-to-end inference pipeline and output contract

Define one reusable inference function for the notebook, HTTP service, qualitative demonstration, and benchmark. The function does not change model thresholds after test evaluation.

نعرّف دالة استدلال واحدة تستخدم في الدفتر والخدمة والعرض وقياس الأداء. لا تغير الدالة حدود النماذج بعد التقييم النهائي.


In [ ]:
M8_DIR = OUTPUT_ROOT / "M8_deployment"
M8_DIR.mkdir(parents=True, exist_ok=True)

def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def encode_png_base64(array_uint8):
    buffer = BytesIO()
    Image.fromarray(array_uint8).save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("ascii")

def m5_segment_candidate(image, box):
    width, height = image.size
    xmin, ymin, xmax, ymax = [float(v) for v in box]
    pad_x = CFG["m5_crop_padding"] * max(xmax - xmin, 1.0)
    pad_y = CFG["m5_crop_padding"] * max(ymax - ymin, 1.0)
    left = max(0, int(math.floor(xmin - pad_x)))
    top = max(0, int(math.floor(ymin - pad_y)))
    right = min(width, int(math.ceil(xmax + pad_x)))
    bottom = min(height, int(math.ceil(ymax + pad_y)))
    crop = image.crop((left, top, right, bottom)).convert("RGB")
    tensor = TF.to_tensor(TF.resize(
        crop, [CFG["m5_imgsz"], CFG["m5_imgsz"]],
        interpolation=InterpolationMode.BILINEAR, antialias=True,
    ))
    tensor = ((tensor - IMAGENET_MEAN) / IMAGENET_STD).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probability = torch.sigmoid(M5_MODEL(tensor))[0, 0].cpu()
    resized = TF.resize(
        probability.unsqueeze(0), [bottom - top, right - left],
        interpolation=InterpolationMode.BILINEAR, antialias=True,
    )[0].numpy()
    return resized, (left, top, right, bottom)

def run_end_to_end(image, include_images=True):
    if not isinstance(image, Image.Image):
        image = Image.open(image)
    image = ImageOps.exif_transpose(image).convert("RGB")
    width, height = image.size
    stage_ms = {}

    synchronize_device(); started = time.perf_counter()
    detection = m4_model.predict(
        source=np.asarray(image), imgsz=CFG["m4_imgsz"],
        conf=M4_OPERATING_POINT["confidence"], iou=M4_OPERATING_POINT["nms_iou"],
        device=0 if DEVICE.type == "cuda" else "cpu", verbose=False,
    )[0]
    synchronize_device(); stage_ms["detection"] = (time.perf_counter() - started) * 1000

    boxes = detection.boxes.xyxy.cpu().numpy() if detection.boxes is not None else np.empty((0, 4))
    confidences = detection.boxes.conf.cpu().numpy() if detection.boxes is not None else np.empty((0,))

    synchronize_device(); started = time.perf_counter()
    union_probability = np.zeros((height, width), dtype=np.float32)
    response_boxes = []
    for box, confidence in zip(boxes, confidences):
        crop_probability, (left, top, right, bottom) = m5_segment_candidate(image, box)
        union_probability[top:bottom, left:right] = np.maximum(
            union_probability[top:bottom, left:right], crop_probability
        )
        response_boxes.append({
            "xyxy": [round(float(v), 3) for v in box],
            "confidence": round(float(confidence), 6),
        })
    synchronize_device(); stage_ms["segmentation"] = (time.perf_counter() - started) * 1000

    binary_mask = (union_probability >= M5_THRESHOLD).astype(np.uint8) * 255
    verdict = "suspected_manipulation" if response_boxes and binary_mask.any() else "no_region_detected"

    synchronize_device(); started = time.perf_counter()
    annotated = image.copy()
    drawer = ImageDraw.Draw(annotated)
    for item in response_boxes:
        drawer.rectangle(item["xyxy"], outline="red", width=max(2, width // 300))
        drawer.text((item["xyxy"][0], item["xyxy"][1]), f"{item['confidence']:.3f}", fill="red")
    if binary_mask.any():
        overlay = Image.new("RGBA", image.size, (0, 0, 0, 0))
        overlay_array = np.asarray(overlay).copy()
        overlay_array[binary_mask > 0] = [255, 0, 0, 100]
        annotated = Image.alpha_composite(annotated.convert("RGBA"), Image.fromarray(overlay_array)).convert("RGB")
    stage_ms["postprocessing"] = (time.perf_counter() - started) * 1000

    payload = {
        "model_version": "ForgeryGuard_AI_Phase2",
        "verdict": verdict,
        "image_width": width,
        "image_height": height,
        "boxes": response_boxes,
        "mask_threshold": M5_THRESHOLD,
        "mask_positive_pixels": int((binary_mask > 0).sum()),
        "stage_latency_ms": {key: round(value, 3) for key, value in stage_ms.items()},
        "total_latency_ms": round(sum(stage_ms.values()), 3),
    }
    if include_images:
        payload["mask_png_base64"] = encode_png_base64(binary_mask)
        annotated_buffer = BytesIO()
        annotated.save(annotated_buffer, format="JPEG", quality=90)
        payload["annotated_jpeg_base64"] = base64.b64encode(annotated_buffer.getvalue()).decode("ascii")
    return payload, annotated, binary_mask

OUTPUT_CONTRACT = {
    "input": "multipart JPG/PNG image or PIL RGB image",
    "output_fields": [
        "model_version", "verdict", "image_width", "image_height", "boxes",
        "mask_threshold", "mask_positive_pixels", "stage_latency_ms", "total_latency_ms",
        "mask_png_base64", "annotated_jpeg_base64",
    ],
    "limitations": "Indicator of possible manipulation; not proof of fraud or legal forgery.",
}
write_json(M8_DIR / "output_contract.json", OUTPUT_CONTRACT)
print(json.dumps(OUTPUT_CONTRACT, indent=2))


## M8 — Step 22: End-to-end qualitative demonstration

Run the complete system on fixed authentic and manipulated test images. Display the submitted image, final annotated result, and union mask, and print the compact JSON contract without embedding large base64 strings in the visible output.

نشغل النظام الكامل على صور اختبار أصلية ومعدلة ثابتة، ثم نعرض الصورة والمدخل النهائي والقناع ونطبع استجابة مختصرة دون عرض النصوص المشفرة الطويلة.


In [ ]:
demo_pairs = PHASE2_PAIRS.loc[PHASE2_PAIRS["split"] == "test"].sample(3, random_state=CFG["seed"])
demo_rows = []
for row in demo_pairs.itertuples(index=False):
    demo_rows.extend([
        (row.source_group, "authentic", row.original_path),
        (row.source_group, "manipulated", row.forged_path),
    ])

fig, axes = plt.subplots(len(demo_rows), 3, figsize=(15, 4 * len(demo_rows)), squeeze=False)
M8_DEMO_RESPONSES = []
for axis_row, (group, expected_type, path) in enumerate(demo_rows):
    with Image.open(path) as image:
        input_image = ImageOps.exif_transpose(image).convert("RGB")
    payload, annotated, binary_mask = run_end_to_end(input_image, include_images=False)
    compact = {key: value for key, value in payload.items()
               if key not in {"mask_png_base64", "annotated_jpeg_base64"}}
    compact.update({"source_group": group, "expected_type": expected_type, "path": path})
    M8_DEMO_RESPONSES.append(compact)
    axes[axis_row, 0].imshow(input_image); axes[axis_row, 0].set_title(f"{group} | {expected_type}")
    axes[axis_row, 1].imshow(annotated); axes[axis_row, 1].set_title(payload["verdict"])
    axes[axis_row, 2].imshow(binary_mask, cmap="gray", vmin=0, vmax=255); axes[axis_row, 2].set_title("Union mask")
    for axis in axes[axis_row]: axis.axis("off")

plt.tight_layout()
fig.savefig(M8_DIR / "end_to_end_demo.png", dpi=170, bbox_inches="tight")
plt.show(); plt.close(fig)
pd.DataFrame(M8_DEMO_RESPONSES).to_json(M8_DIR / "demo_responses.json", orient="records", indent=2)
display(pd.DataFrame(M8_DEMO_RESPONSES)[[
    "source_group", "expected_type", "verdict", "mask_positive_pixels", "total_latency_ms"
]])


## M8 — Step 23: Export the principal detector to ONNX and verify parity

Export the selected M4 checkpoint to ONNX with a fixed 640×640 input. Compare corresponding raw model outputs for one deterministic validation image and report maximum and mean absolute differences. The declared parity target is a maximum absolute difference no greater than 0.001.

نصدر نموذج الكشف المختار إلى أونكس بمدخل ثابت، ثم نقارن المخرجات الخام المتناظرة على صورة تحقق ثابتة ونحسب أكبر ومتوسط فرق مطلق. حد القبول المعلن هو 0.001 لأكبر فرق.


In [ ]:
import onnx
import onnxruntime as ort
from ultralytics.data.augment import LetterBox

exported_path = Path(m4_model.export(
    format="onnx", imgsz=CFG["m4_imgsz"], opset=CFG["onnx_opset"],
    dynamic=False, simplify=False, half=False, device="cpu",
))
M4_ONNX = M8_DIR / "m4_detector.onnx"
if exported_path.resolve() != M4_ONNX.resolve():
    shutil.copy2(exported_path, M4_ONNX)
onnx.checker.check_model(onnx.load(M4_ONNX))

parity_row = PHASE2_PAIRS.loc[PHASE2_PAIRS["split"] == "val"].sort_values("source_group").iloc[0]
with Image.open(parity_row.forged_path) as image:
    parity_rgb = np.asarray(image.convert("RGB"))
letterbox = LetterBox(new_shape=(CFG["m4_imgsz"], CFG["m4_imgsz"]), auto=False, stride=32)
prepared = letterbox(image=parity_rgb)
model_input = torch.from_numpy(prepared.transpose(2, 0, 1)).float().div(255).unsqueeze(0)

m4_model.model.to("cpu").eval()
with torch.no_grad():
    pytorch_raw = m4_model.model(model_input)
if isinstance(pytorch_raw, (tuple, list)):
    pytorch_raw = pytorch_raw[0]
pytorch_raw = pytorch_raw.detach().cpu().numpy()

ort_session = ort.InferenceSession(str(M4_ONNX), providers=["CPUExecutionProvider"])
onnx_raw = ort_session.run(None, {ort_session.get_inputs()[0].name: model_input.numpy()})[0]
assert pytorch_raw.shape == onnx_raw.shape, (pytorch_raw.shape, onnx_raw.shape)

absolute_difference = np.abs(pytorch_raw - onnx_raw)
ONNX_PARITY = {
    "sample_source_group": str(parity_row.source_group),
    "pytorch_shape": list(pytorch_raw.shape),
    "onnx_shape": list(onnx_raw.shape),
    "max_absolute_difference": float(absolute_difference.max()),
    "mean_absolute_difference": float(absolute_difference.mean()),
    "target_max_absolute_difference": CFG["acceptance"]["onnx_max_abs_difference"],
    "passed": bool(absolute_difference.max() <= CFG["acceptance"]["onnx_max_abs_difference"]),
    "onnx_sha256": sha256_file(M4_ONNX),
}
write_json(M8_DIR / "onnx_parity.json", ONNX_PARITY)
print(json.dumps(ONNX_PARITY, indent=2))

m4_model.model.to(DEVICE)
M5_MODEL.eval()
example_m5_input = torch.zeros(1, 3, CFG["m5_imgsz"], CFG["m5_imgsz"], device=DEVICE)
with torch.no_grad():
    m5_reference_output = M5_MODEL(example_m5_input)
M5_TORCHSCRIPT = M8_DIR / "m5_segmenter.ts"
traced_m5 = torch.jit.trace(M5_MODEL, example_m5_input)
traced_m5.save(str(M5_TORCHSCRIPT))
with torch.no_grad():
    m5_script_output = traced_m5(example_m5_input)
M5_SCRIPT_PARITY = {
    "max_absolute_difference": float((m5_reference_output - m5_script_output).abs().max().item()),
    "passed": bool(torch.allclose(m5_reference_output, m5_script_output, atol=1e-5, rtol=1e-5)),
    "torchscript_sha256": sha256_file(M5_TORCHSCRIPT),
}
write_json(M8_DIR / "m5_torchscript_parity.json", M5_SCRIPT_PARITY)
print("M5 TorchScript parity:", json.dumps(M5_SCRIPT_PARITY, indent=2))

M4_DEPLOYMENT_PT = M8_DIR / "m4_detector.pt"
if Path(M4_BEST).resolve() != M4_DEPLOYMENT_PT.resolve():
    shutil.copy2(M4_BEST, M4_DEPLOYMENT_PT)


## M8 — Step 24: FastAPI HTTP service and smoke test

Expose `GET /health` and `POST /predict`. The prediction endpoint accepts one JPG or PNG file and returns the documented JSON response. `TestClient` exercises the real HTTP route without requiring a public tunnel or third-party deployment key.

نوفر مساري فحص الصحة والتنبؤ. يستقبل مسار التنبؤ صورة واحدة ويعيد استجابة جيسون الموثقة. يختبر العميل المحلي مسار إتش تي تي بي نفسه دون الحاجة إلى رابط عام أو مفتاح خدمة خارجية.


In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.testclient import TestClient

app = FastAPI(
    title="ForgeryGuard AI",
    version="2.0",
    description="Academic screening API for copy-move manipulation indicators.",
)

@app.get("/health")
def health():
    return {
        "status": "ok",
        "device": str(DEVICE),
        "m4_checkpoint_sha256": sha256_file(M4_BEST),
        "m5_checkpoint_sha256": sha256_file(M5_BEST),
    }

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    if file.content_type not in {"image/jpeg", "image/png"}:
        raise HTTPException(status_code=415, detail="Only JPG and PNG images are supported.")
    content = await file.read()
    try:
        image = Image.open(BytesIO(content))
        image.load()
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Invalid image: {type(exc).__name__}") from exc
    payload, _, _ = run_end_to_end(image, include_images=True)
    return payload

api_test_row = PHASE2_PAIRS.loc[PHASE2_PAIRS["split"] == "test"].sort_values("source_group").iloc[0]
client = TestClient(app)
health_response = client.get("/health")
assert health_response.status_code == 200
with open(api_test_row.forged_path, "rb") as stream:
    prediction_response = client.post(
        "/predict", files={"file": (Path(api_test_row.forged_path).name, stream, "image/tiff")}
    )

# The endpoint intentionally accepts JPG/PNG only. Convert the TIFF test image to PNG for the valid request.
with Image.open(api_test_row.forged_path) as image:
    png_buffer = BytesIO()
    image.convert("RGB").save(png_buffer, format="PNG")
valid_response = client.post(
    "/predict", files={"file": ("test.png", png_buffer.getvalue(), "image/png")}
)
assert prediction_response.status_code == 415
assert valid_response.status_code == 200
API_SMOKE_TEST = {
    "health_status_code": health_response.status_code,
    "unsupported_tiff_status_code": prediction_response.status_code,
    "valid_png_status_code": valid_response.status_code,
    "valid_response_fields": sorted(valid_response.json().keys()),
    "passed": True,
}
write_json(M8_DIR / "api_smoke_test.json", API_SMOKE_TEST)
print(json.dumps(API_SMOKE_TEST, indent=2))

SERVICE_SOURCE = r'''from io import BytesIO
from pathlib import Path
import base64
import math
import time

import numpy as np
from PIL import Image, ImageOps, ImageDraw
import torch
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
from ultralytics import YOLO
from fastapi import FastAPI, UploadFile, File, HTTPException

BASE_DIR = Path(__file__).resolve().parent
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
M4_IMAGE_SIZE = 640
M5_IMAGE_SIZE = 256
M4_CONFIDENCE = __M4_CONFIDENCE__
M4_NMS_IOU = __M4_NMS_IOU__
M5_THRESHOLD = __M5_THRESHOLD__
M5_CROP_PADDING = __M5_CROP_PADDING__
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225])[:, None, None]

M4_MODEL = YOLO(str(BASE_DIR / "m4_detector.pt"))
M5_MODEL = torch.jit.load(str(BASE_DIR / "m5_segmenter.ts"), map_location=DEVICE).eval()

def encode_png_base64(array_uint8):
    buffer = BytesIO()
    Image.fromarray(array_uint8).save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("ascii")

def segment_candidate(image, box):
    width, height = image.size
    xmin, ymin, xmax, ymax = [float(v) for v in box]
    pad_x = M5_CROP_PADDING * max(xmax - xmin, 1.0)
    pad_y = M5_CROP_PADDING * max(ymax - ymin, 1.0)
    left = max(0, int(math.floor(xmin - pad_x)))
    top = max(0, int(math.floor(ymin - pad_y)))
    right = min(width, int(math.ceil(xmax + pad_x)))
    bottom = min(height, int(math.ceil(ymax + pad_y)))
    crop = image.crop((left, top, right, bottom)).convert("RGB")
    tensor = TF.to_tensor(TF.resize(
        crop, [M5_IMAGE_SIZE, M5_IMAGE_SIZE],
        interpolation=InterpolationMode.BILINEAR, antialias=True,
    ))
    tensor = ((tensor - IMAGENET_MEAN) / IMAGENET_STD).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probability = torch.sigmoid(M5_MODEL(tensor))[0, 0].cpu()
    resized = TF.resize(
        probability.unsqueeze(0), [bottom - top, right - left],
        interpolation=InterpolationMode.BILINEAR, antialias=True,
    )[0].numpy()
    return resized, (left, top, right, bottom)

def infer(image):
    image = ImageOps.exif_transpose(image).convert("RGB")
    width, height = image.size
    total_started = time.perf_counter()
    started = time.perf_counter()
    detection = M4_MODEL.predict(
        source=np.asarray(image), imgsz=M4_IMAGE_SIZE, conf=M4_CONFIDENCE,
        iou=M4_NMS_IOU, device=0 if DEVICE.type == "cuda" else "cpu", verbose=False,
    )[0]
    detection_ms = (time.perf_counter() - started) * 1000
    boxes = detection.boxes.xyxy.cpu().numpy() if detection.boxes is not None else np.empty((0, 4))
    confidences = detection.boxes.conf.cpu().numpy() if detection.boxes is not None else np.empty((0,))
    union_probability = np.zeros((height, width), dtype=np.float32)
    response_boxes = []
    started = time.perf_counter()
    for box, confidence in zip(boxes, confidences):
        probability, (left, top, right, bottom) = segment_candidate(image, box)
        union_probability[top:bottom, left:right] = np.maximum(
            union_probability[top:bottom, left:right], probability
        )
        response_boxes.append({
            "xyxy": [round(float(v), 3) for v in box],
            "confidence": round(float(confidence), 6),
        })
    segmentation_ms = (time.perf_counter() - started) * 1000
    started = time.perf_counter()
    binary_mask = (union_probability >= M5_THRESHOLD).astype(np.uint8) * 255
    annotated = image.copy()
    drawer = ImageDraw.Draw(annotated)
    for item in response_boxes:
        drawer.rectangle(item["xyxy"], outline="red", width=max(2, width // 300))
        drawer.text((item["xyxy"][0], item["xyxy"][1]), f"{item['confidence']:.3f}", fill="red")
    if binary_mask.any():
        overlay_array = np.zeros((height, width, 4), dtype=np.uint8)
        overlay_array[binary_mask > 0] = [255, 0, 0, 100]
        annotated = Image.alpha_composite(
            annotated.convert("RGBA"), Image.fromarray(overlay_array, mode="RGBA")
        ).convert("RGB")
    annotated_buffer = BytesIO()
    annotated.save(annotated_buffer, format="JPEG", quality=90)
    postprocessing_ms = (time.perf_counter() - started) * 1000
    return {
        "model_version": "ForgeryGuard_AI_Phase2",
        "verdict": "suspected_manipulation" if response_boxes and binary_mask.any() else "no_region_detected",
        "image_width": width,
        "image_height": height,
        "boxes": response_boxes,
        "mask_threshold": M5_THRESHOLD,
        "mask_positive_pixels": int((binary_mask > 0).sum()),
        "stage_latency_ms": {
            "detection": round(detection_ms, 3),
            "segmentation": round(segmentation_ms, 3),
            "postprocessing": round(postprocessing_ms, 3),
        },
        "total_latency_ms": round((time.perf_counter() - total_started) * 1000, 3),
        "mask_png_base64": encode_png_base64(binary_mask),
        "annotated_jpeg_base64": base64.b64encode(annotated_buffer.getvalue()).decode("ascii"),
    }

app = FastAPI(title="ForgeryGuard AI", version="2.0")

@app.get("/health")
def health():
    return {"status": "ok", "device": str(DEVICE)}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    if file.content_type not in {"image/jpeg", "image/png"}:
        raise HTTPException(status_code=415, detail="Only JPG and PNG images are supported.")
    try:
        image = Image.open(BytesIO(await file.read()))
        image.load()
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Invalid image: {type(exc).__name__}") from exc
    return infer(image)
'''

SERVICE_SOURCE = (SERVICE_SOURCE
    .replace("__M4_CONFIDENCE__", repr(float(M4_OPERATING_POINT["confidence"])))
    .replace("__M4_NMS_IOU__", repr(float(M4_OPERATING_POINT["nms_iou"])))
    .replace("__M5_THRESHOLD__", repr(float(M5_THRESHOLD)))
    .replace("__M5_CROP_PADDING__", repr(float(CFG["m5_crop_padding"]))))
(M8_DIR / "phase2_api.py").write_text(SERVICE_SOURCE, encoding="utf-8")
compile(SERVICE_SOURCE, "phase2_api.py", "exec")
print("Standalone service source:", M8_DIR / "phase2_api.py")


## M8 — Step 25: End-to-end latency, throughput, and memory benchmark

Warm up the complete PyTorch pipeline, then process a fixed test subset one image at a time on the documented hardware. Report p50, p95, and p99 latency, throughput, peak allocated GPU memory, reserved GPU memory, and process RSS. The benchmark includes detection, segmentation, and postprocessing.

نسخن مسار الاستدلال الكامل ثم نعالج عينة اختبار ثابتة بصورة واحدة في كل مرة على العتاد المسجل. نحسب أزمنة المنتصفات والنسب العليا ومعدل المعالجة وذاكرة الجهاز والعملية، ويشمل القياس الكشف والتقسيم والمعالجة النهائية.


In [ ]:
import psutil

benchmark_frame = PHASE2_PAIRS.loc[PHASE2_PAIRS["split"] == "test"].sample(
    min(CFG["m8_benchmark_images"], (PHASE2_PAIRS["split"] == "test").sum()),
    random_state=CFG["seed"],
).reset_index(drop=True)

warmup_path = benchmark_frame.iloc[0].forged_path
for _ in range(CFG["m8_warmup_runs"]):
    run_end_to_end(warmup_path, include_images=False)

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
process = psutil.Process(os.getpid())
rss_before = process.memory_info().rss
benchmark_rows = []
benchmark_started = time.perf_counter()

for row in tqdm(benchmark_frame.itertuples(index=False), total=len(benchmark_frame), desc="M8 benchmark"):
    synchronize_device(); started = time.perf_counter()
    payload, _, _ = run_end_to_end(row.forged_path, include_images=False)
    synchronize_device(); elapsed_ms = (time.perf_counter() - started) * 1000
    benchmark_rows.append({
        "source_group": row.source_group,
        "latency_ms": elapsed_ms,
        **{f"stage_{key}_ms": value for key, value in payload["stage_latency_ms"].items()},
        "detected_boxes": len(payload["boxes"]),
    })

total_seconds = time.perf_counter() - benchmark_started
M8_BENCHMARK_ROWS = pd.DataFrame(benchmark_rows)
latencies = M8_BENCHMARK_ROWS["latency_ms"].to_numpy()
M8_BENCHMARK = {
    "hardware": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else platform.processor(),
    "device": str(DEVICE),
    "batch_size": 1,
    "warmup_runs": CFG["m8_warmup_runs"],
    "measured_images": len(M8_BENCHMARK_ROWS),
    "p50_latency_ms": float(np.percentile(latencies, 50)),
    "p95_latency_ms": float(np.percentile(latencies, 95)),
    "p99_latency_ms": float(np.percentile(latencies, 99)),
    "throughput_images_per_second": float(len(M8_BENCHMARK_ROWS) / total_seconds),
    "process_rss_before_mb": float(rss_before / 1024**2),
    "process_rss_after_mb": float(process.memory_info().rss / 1024**2),
    "peak_gpu_allocated_mb": float(torch.cuda.max_memory_allocated() / 1024**2) if DEVICE.type == "cuda" else None,
    "peak_gpu_reserved_mb": float(torch.cuda.max_memory_reserved() / 1024**2) if DEVICE.type == "cuda" else None,
    "conditions": "Complete PyTorch M4→M5 pipeline, batch size 1, encoded images disabled during timing.",
}
M8_BENCHMARK_ROWS.to_csv(M8_DIR / "benchmark_rows.csv", index=False)
write_json(M8_DIR / "benchmark_summary.json", M8_BENCHMARK)
display(M8_BENCHMARK_ROWS.describe(include="all"))
print(json.dumps(M8_BENCHMARK, indent=2))


## M8 — Step 26: Model Card and go/no-go decision

Compare the locked test and benchmark results with the predeclared criteria. A failed criterion produces a no-go decision for autonomous use while preserving the system as an academic demonstration or human-review aid. The Model Card records intended use, out-of-scope use, data, metrics, limitations, privacy, licensing status, and ethical constraints.

نقارن نتائج الاختبار والقياس بالمعايير المثبتة. يؤدي فشل أي معيار إلى قرار عدم الاستخدام المستقل مع بقاء النظام عرضًا أكاديميًا أو أداة مساعدة للمراجع البشري. توثق بطاقة النموذج الاستخدام والبيانات والمقاييس والقيود والخصوصية والترخيص.


In [ ]:
GO_NO_GO_CHECKS = {
    "m4_test_map50": M4_TEST_METRICS["map50"] >= CFG["acceptance"]["m4_min_test_map50"],
    "m4_test_recall": M4_TEST_METRICS["recall_at_operating_point"] >= CFG["acceptance"]["m4_min_test_recall"],
    "m5_test_dice": M5_TEST_METRICS["dice"] >= CFG["acceptance"]["m5_min_test_dice"],
    "m5_test_iou": M5_TEST_METRICS["iou"] >= CFG["acceptance"]["m5_min_test_iou"],
    "onnx_parity": ONNX_PARITY["passed"],
    "m8_p95_latency": M8_BENCHMARK["p95_latency_ms"] <= CFG["acceptance"]["m8_max_p95_ms"],
    "m8_throughput": M8_BENCHMARK["throughput_images_per_second"] >= CFG["acceptance"]["m8_min_throughput_images_per_second"],
}
GO_NO_GO = "GO_FOR_AUTONOMOUS_USE" if all(GO_NO_GO_CHECKS.values()) else "NO_GO_FOR_AUTONOMOUS_USE"
FAILED_CRITERIA = [key for key, passed in GO_NO_GO_CHECKS.items() if not passed]

MODEL_CARD = f'''# ForgeryGuard AI — Model Card

## Model details
- System: YOLO11n copy-move region detector followed by a ResNet18 U-Net crop segmenter.
- Version: Phase 2 official run.
- Dataset: DEFACTO Copy-Move forged images with MSCOCO originals; corrected source-group split.
- Effective source pairs: {len(PHASE2_PAIRS):,}.

## Intended use
Academic study and human-reviewed screening of possible copy-move manipulation in digital images.

## Out-of-scope use
Legal proof of fraud, autonomous enforcement, surveillance, identity decisions, or manipulation types outside the evaluated copy-move scope.

## Test results
- M4 mAP50: {M4_TEST_METRICS['map50']:.6f}
- M4 mAP50–95: {M4_TEST_METRICS['map50_95']:.6f}
- M4 recall at confidence {M4_OPERATING_POINT['confidence']:.2f}: {M4_TEST_METRICS['recall_at_operating_point']:.6f}
- M5 Dice at threshold {M5_THRESHOLD:.2f}: {M5_TEST_METRICS['dice']:.6f}
- M5 mean IoU: {M5_TEST_METRICS['iou']:.6f}
- Crop-level mask AP50: {M5_MASK_AP50:.6f}
- Crop-level mask mAP50–95: {M5_MASK_MAP50_95:.6f}
- End-to-end p95 latency: {M8_BENCHMARK['p95_latency_ms']:.3f} ms
- End-to-end throughput: {M8_BENCHMARK['throughput_images_per_second']:.3f} images/s

## Decision
- Decision: {GO_NO_GO}
- Failed criteria: {', '.join(FAILED_CRITERIA) if FAILED_CRITERIA else 'None'}
- Permitted release posture: academic demonstration and assisted screening with human review.

## Limitations
Small or visually subtle manipulated regions may be missed. The crop segmenter depends on detector proposals, and performance may shift under new compression, editing tools, image sources, or manipulation types.

## Privacy and licensing
The dataset may include identifiable people and personal content. Dataset access and redistribution rights remain subject to DEFACTO, MSCOCO, upstream image-owner, and institutional terms. Restricted images are not included in the repository or release archive.

## Ethical statement
The output is an indicator of possible manipulation, not proof of forgery or fraud. Human review is required for consequential decisions.
'''

(M8_DIR / "MODEL_CARD.md").write_text(MODEL_CARD, encoding="utf-8")
GO_NO_GO_RECORD = {
    "decision": GO_NO_GO,
    "checks": GO_NO_GO_CHECKS,
    "failed_criteria": FAILED_CRITERIA,
    "release_scope": "Academic demonstration and human-review screening only.",
}
write_json(M8_DIR / "go_no_go_decision.json", GO_NO_GO_RECORD)
display(pd.DataFrame([
    {"criterion": key, "passed": value} for key, value in GO_NO_GO_CHECKS.items()
]))
display(Markdown(MODEL_CARD))


## Shared — Step 27: W&B experiment evidence

Create one Phase 2 run, log the frozen configuration, M4/M5/M8 summary metrics, and epoch histories, then print the run URL when online. If the run is offline, the final checklist remains pending until the saved run is synchronized.

ننشىء تجربة واحدة للمرحلة الثانية ونسجل الإعدادات والمقاييس ونتائج الحقب، ثم نطبع الرابط عند الاتصال. إذا كان التسجيل محليًا يبقى بند التتبع معلقًا حتى تتم المزامنة.


In [ ]:
wandb_run = wandb.init(
    project="ForgeryGuard-AI-Phase2",
    name="M4_M5_M8_official",
    config=CFG,
    tags=["Phase2", "M4", "M5", "M8", "official"],
    mode=WANDB_MODE,
    reinit=True,
)

if (M4_PROJECT / "detector_official" / "results.csv").is_file():
    m4_history = pd.read_csv(M4_PROJECT / "detector_official" / "results.csv")
    for epoch_index, row in m4_history.iterrows():
        wandb_run.log({f"M4/train/{key.strip()}": float(value)
                       for key, value in row.items() if pd.notna(value)}, step=int(epoch_index))

if not M5_HISTORY.empty:
    for _, row in M5_HISTORY.iterrows():
        wandb_run.log({f"M5/{key}": float(value) for key, value in row.items()})

WANDB_SUMMARY = {
    "M4/test_mAP50": M4_TEST_METRICS["map50"],
    "M4/test_mAP50_95": M4_TEST_METRICS["map50_95"],
    "M4/test_precision": M4_TEST_METRICS["precision_at_operating_point"],
    "M4/test_recall": M4_TEST_METRICS["recall_at_operating_point"],
    "M5/test_dice": M5_TEST_METRICS["dice"],
    "M5/test_iou": M5_TEST_METRICS["iou"],
    "M5/test_mask_AP50": M5_MASK_AP50,
    "M5/test_mask_mAP50_95": M5_MASK_MAP50_95,
    "M8/onnx_max_abs_difference": ONNX_PARITY["max_absolute_difference"],
    "M8/p95_latency_ms": M8_BENCHMARK["p95_latency_ms"],
    "M8/throughput_images_per_second": M8_BENCHMARK["throughput_images_per_second"],
}
wandb_run.log(WANDB_SUMMARY)
WANDB_RUN_URL = wandb_run.url if WANDB_MODE == "online" else None
WANDB_RUN_PATH = wandb_run.path
wandb_run.finish()

WANDB_RECORD = {"mode": WANDB_MODE, "run_path": WANDB_RUN_PATH, "run_url": WANDB_RUN_URL,
                "summary": WANDB_SUMMARY}
write_json(OUTPUT_ROOT / "wandb_record.json", WANDB_RECORD)
print(json.dumps(WANDB_RECORD, indent=2))


## Shared — Step 28: Reproducibility manifest, pinned requirements, and run instructions

Record dataset and checkpoint hashes, environment versions, thresholds, hardware, and artifact paths. Generate pinned requirements and concise service instructions. A second independent training run is not required for Phase 2; reproducibility is demonstrated through the fixed configuration, committed manifests, hashes, checkpoints, W&B evidence, and executable instructions.

نسجل بصمات البيانات والنماذج وإصدارات البيئة والحدود والعتاد والمسارات، وننشئ ملف المتطلبات وتعليمات الخدمة. لا يلزم تدريب مستقل ثانٍ للمرحلة الثانية؛ تثبت قابلية الإعادة من خلال الإعداد الموحد والبصمات ونقاط الحفظ والتتبع وتعليمات التشغيل.


In [ ]:
packages_to_pin = [
    "torch", "torchvision", "ultralytics", "wandb", "onnx", "onnxruntime",
    "fastapi", "uvicorn", "python-multipart", "httpx", "numpy", "pandas",
    "Pillow", "matplotlib", "PyYAML", "psutil",
]
requirements_lines = []
for package in packages_to_pin:
    try:
        requirements_lines.append(f"{package}=={metadata.version(package)}")
    except metadata.PackageNotFoundError:
        requirements_lines.append(f"# {package}: not installed")
REQUIREMENTS_TEXT = "\n".join(requirements_lines) + "\n"
(OUTPUT_ROOT / "requirements.txt").write_text(REQUIREMENTS_TEXT, encoding="utf-8")

RUN_INSTRUCTIONS = f'''# ForgeryGuard AI — Phase 2 run instructions

1. Attach the project dataset and DEFACTO Copy-Move inputs in Kaggle.
2. Enable a GPU and Internet, or attach the official pretrained weights.
3. Add `WANDB_API_KEY` through Kaggle Secrets.
4. Run the notebook cells in order.
5. Complete the manual review fields in Steps 8B, 13B, and 20B using the displayed evidence.
6. Save the Kaggle version with outputs and download the final ZIP.

The notebook tests `GET /health` and `POST /predict` with FastAPI TestClient. The final archive contains a standalone service module. Start it with:

```bash
cd M8_deployment
uvicorn phase2_api:app --host 0.0.0.0 --port 8000
```

Checkpoints:
- M4: {M4_BEST}
- M5: {M5_BEST}
- ONNX: {M4_ONNX}
'''
(OUTPUT_ROOT / "RUN_INSTRUCTIONS.md").write_text(RUN_INSTRUCTIONS, encoding="utf-8")

ENVIRONMENT_MANIFEST = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "run_owner": RUN_OWNER,
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "m1_source_manifest_sha256": sha256_file(MANIFEST_PATH),
    "m1_effective_manifest_sha256": sha256_file(EFFECTIVE_MANIFEST_PATH),
    "m1_manifest_semantic_sha256": M1_MANIFEST_SEMANTIC_SHA256,
    "bounding_boxes_sha256": sha256_file(BBOX_PATH),
    "m4_checkpoint_sha256": sha256_file(M4_BEST),
    "m5_checkpoint_sha256": sha256_file(M5_BEST),
    "m4_onnx_sha256": sha256_file(M4_ONNX),
    "config_sha256": hashlib.sha256(json.dumps(CFG, sort_keys=True).encode()).hexdigest(),
    "effective_source_pairs": len(PHASE2_PAIRS),
    "m4_operating_point": M4_OPERATING_POINT,
    "m5_mask_threshold": M5_THRESHOLD,
    "wandb": WANDB_RECORD,
}
write_json(OUTPUT_ROOT / "environment_manifest.json", ENVIRONMENT_MANIFEST)

print("Pinned requirements:\n" + REQUIREMENTS_TEXT)
print(json.dumps(ENVIRONMENT_MANIFEST, indent=2))


## Shared — Step 29: Consolidated reviewer summary

Display the exact values needed for the report: split sizes, operating thresholds, M4 metrics, M5 metrics, mask AP, ONNX parity, system performance, W&B link, and go/no-go decision. The summary is stored as JSON and shown in the notebook output so the report can be prepared from the executed notebook alone.

نعرض القيم اللازمة للتقرير داخل مخرجات الدفتر: أحجام التقسيم والحدود ومقاييس الكشف والتقسيم ومتوسط دقة الأقنعة وتطابق التصدير وأداء النظام ورابط التتبع والقرار النهائي. تُحفظ الخلاصة وتظهر كاملة لتسهيل إعداد التقرير من الدفتر المنفذ.


In [ ]:
PHASE2_SUMMARY = {
    "project": "ForgeryGuard AI",
    "effective_source_pairs": len(PHASE2_PAIRS),
    "split_counts": split_counts.to_dict(),
    "m4": {
        "operating_point": M4_OPERATING_POINT,
        "test_metrics": M4_TEST_METRICS,
        "per_class_ap": pd.read_csv(OUTPUT_ROOT / "m4_per_class_ap.csv").to_dict(orient="records"),
        "recall_by_size": recall_by_size.to_dict(orient="records"),
        "false_positive_breakdown": fp_breakdown.to_dict(orient="records"),
    },
    "m5": {
        "selected_threshold": M5_THRESHOLD,
        "otsu_baseline": OTSU_BASELINE,
        "validation_metrics": M5_VAL_METRICS,
        "test_metrics": M5_TEST_METRICS,
        "mask_ap": M5_MASK_AP_METRICS,
    },
    "m8": {
        "onnx_parity": ONNX_PARITY,
        "api_smoke_test": API_SMOKE_TEST,
        "benchmark": M8_BENCHMARK,
        "go_no_go": GO_NO_GO_RECORD,
    },
    "wandb": WANDB_RECORD,
}
write_json(OUTPUT_ROOT / "phase2_summary.json", PHASE2_SUMMARY)

summary_table = pd.DataFrame([
    ["M4", "Test mAP50", M4_TEST_METRICS["map50"]],
    ["M4", "Test mAP50–95", M4_TEST_METRICS["map50_95"]],
    ["M4", "Test recall", M4_TEST_METRICS["recall_at_operating_point"]],
    ["M5", "Test Dice", M5_TEST_METRICS["dice"]],
    ["M5", "Test mean IoU", M5_TEST_METRICS["iou"]],
    ["M5", "Mask AP50", M5_MASK_AP50],
    ["M5", "Mask mAP50–95", M5_MASK_MAP50_95],
    ["M8", "ONNX max absolute difference", ONNX_PARITY["max_absolute_difference"]],
    ["M8", "p95 latency (ms)", M8_BENCHMARK["p95_latency_ms"]],
    ["M8", "Throughput (images/s)", M8_BENCHMARK["throughput_images_per_second"]],
], columns=["milestone", "metric", "value"])
display(summary_table)
print("W&B URL:", WANDB_RUN_URL)
print("Go/No-Go decision:", GO_NO_GO)
print("Failed criteria:", FAILED_CRITERIA)


## Shared — Step 30: Submission checklist and final archive

The checklist distinguishes technical execution from submission readiness. It requires the manual annotation review, two written failure analyses, checkpoints, all requested metrics, ONNX parity evidence, API smoke test, benchmark, Model Card, W&B evidence, and reproducibility files. The archive excludes raw restricted images and includes only code-generated evidence and model artifacts.

يفصل الفحص بين اكتمال التنفيذ وجاهزية التسليم. يتطلب المراجعة اليدوية وتحليلي الفشل ونقاط الحفظ والمقاييس وتطابق التصدير واختبار الخدمة وقياس الأداء وبطاقة النموذج والتتبع وملفات إعادة التشغيل. لا يتضمن الأرشيف صور البيانات المقيدة.


In [ ]:
FINAL_CHECKLIST = {
    "corrected_m1_manifest_verified": M1_MANIFEST_SEMANTIC_SHA256 == EXPECTED_M1_SEMANTIC_SHA256,
    "source_group_leakage_absent": PHASE2_PAIRS.groupby("source_group")["split"].nunique().max() == 1,
    "annotation_integrity_audit_passed": issues_frame.empty,
    "manual_box_review_completed": bool(M4_BOX_REVIEW_COMPLETED),
    "manual_mask_review_completed": bool(M5_MASK_REVIEW_COMPLETED),
    "m4_checkpoint_saved": M4_BEST.is_file(),
    "m4_validation_sweep_saved": (OUTPUT_ROOT / "m4_threshold_sweep.csv").is_file(),
    "m4_test_metrics_saved": (OUTPUT_ROOT / "m4_test_metrics.json").is_file(),
    "m4_error_analysis_saved": (OUTPUT_ROOT / "m4_false_positive_breakdown.csv").is_file(),
    "m4_written_failure_analysis_completed": bool(M4_FAILURE_ANALYSIS_COMPLETED),
    "m5_checkpoint_saved": M5_BEST.is_file(),
    "m5_test_metrics_saved": (M5_DIR / "test_metrics.json").is_file(),
    "m5_mask_ap_saved": (M5_DIR / "mask_ap_metrics.json").is_file(),
    "m5_written_failure_analysis_completed": bool(M5_FAILURE_ANALYSIS_COMPLETED),
    "m8_end_to_end_demo_saved": (M8_DIR / "end_to_end_demo.png").is_file(),
    "m8_onnx_export_saved": M4_ONNX.is_file(),
    "m8_onnx_parity_recorded": (M8_DIR / "onnx_parity.json").is_file(),
    "m8_api_smoke_test_passed": API_SMOKE_TEST["passed"],
    "standalone_fastapi_module_saved": (M8_DIR / "phase2_api.py").is_file(),
    "m8_benchmark_saved": (M8_DIR / "benchmark_summary.json").is_file(),
    "model_card_saved": (M8_DIR / "MODEL_CARD.md").is_file(),
    "go_no_go_recorded": (M8_DIR / "go_no_go_decision.json").is_file(),
    "wandb_online_link_available": bool(WANDB_RUN_URL),
    "requirements_saved": (OUTPUT_ROOT / "requirements.txt").is_file(),
    "environment_manifest_saved": (OUTPUT_ROOT / "environment_manifest.json").is_file(),
    "run_instructions_saved": (OUTPUT_ROOT / "RUN_INSTRUCTIONS.md").is_file(),
}
write_json(OUTPUT_ROOT / "final_checklist.json", FINAL_CHECKLIST)
display(pd.DataFrame([
    {"requirement": key, "complete": value} for key, value in FINAL_CHECKLIST.items()
]))

pending = [key for key, complete in FINAL_CHECKLIST.items() if not complete]

# Build a source/evidence package without raw dataset links or image-containing audit panels.
PACKAGE_ROOT = Path("/kaggle/working/ForgeryGuard_AI_Phase2_Source")
if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)
PACKAGE_ROOT.mkdir(parents=True)
excluded_directories = {"m4_yolo_dataset", "manual_review_panels"}
excluded_image_suffixes = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
for source in OUTPUT_ROOT.rglob("*"):
    relative = source.relative_to(OUTPUT_ROOT)
    if any(part in excluded_directories for part in relative.parts):
        continue
    if source.is_symlink() or source.is_dir() or source.suffix.lower() in excluded_image_suffixes:
        continue
    destination = PACKAGE_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

archive_base = Path("/kaggle/working") / "ForgeryGuard_AI_Phase2_source"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=PACKAGE_ROOT))

print("Technical pipeline executed:", True)
print("Submission-ready checklist:", "PASS" if not pending else "PENDING")
print("Pending items:", pending)
print("Output folder:", OUTPUT_ROOT)
print("Final evidence archive:", archive_path)
print("Save a Kaggle version with all outputs before closing the session.")

assert not pending, (
    "Complete the pending manual/W&B items shown above, then rerun their cells and this final cell."
)


## Phase 1 reminder before final project submission

Phase 2 completion does not close the documented Phase 1 corrections. Before the final project submission, commit the Phase 1 correction report and evidence, verify the corrected-manifest hash and W&B/GitHub links, and ensure the report describes the decode audit, near-duplicate remediation, per-class metrics, calibration interpretation, quantitative error analysis, pinned requirements, and reproducibility evidence.

إكمال المرحلة الثانية لا يلغي تصحيحات المرحلة الأولى. قبل التسليم النهائي يجب رفع تقرير التصحيح وأدلته، والتحقق من بصمة التقسيم والروابط، وتوثيق فحص فك الترميز ومعالجة التشابه القريب ومقاييس الفئات والمعايرة وتحليل الأخطاء والمتطلبات وأدلة إعادة التشغيل.
